In [ ]:
install.packages("SoupX")
library(SoupX)

In [ ]:
# library(SoupX)
library(Seurat)
library(dplyr)

In [ ]:
# Method 1: Use forward slashes (recommended)
root_dir <- "./data"


# Check whether the path exists
file.exists(root_dir)           # Must return TRUE
dir.exists(root_dir)            # Must return TRUE

# List the contents of this directory
list.dirs(root_dir, recursive = FALSE)
list.files(root_dir)

In [ ]:
if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager")
BiocManager::install("DropletUtils")

In [ ]:
library(SoupX)
library(Seurat)
library(dplyr)
library(Matrix)

# ========== Paths ==========
root_dir <- "./data"

if (!dir.exists(root_dir)) {
  stop("Path does not exist! Please check: ", root_dir)
}

# ========== Custom function to read 10X data ==========
read10x_custom <- function(data.dir) {
  mat <- Matrix::readMM(gzfile(file.path(data.dir, "matrix.mtx.gz")))
  
  barcodes <- read.table(
    gzfile(file.path(data.dir, "barcodes.tsv.gz")),
    header = FALSE, stringsAsFactors = FALSE, sep = "\t"
  )[, 1]
  
  features <- read.table(
    gzfile(file.path(data.dir, "features.tsv.gz")),
    header = FALSE, stringsAsFactors = FALSE, sep = "\t"
  )
  
  if (ncol(features) >= 2) {
    rownames(mat) <- make.names(features[, 2], unique = TRUE)
  } else {
    rownames(mat) <- make.names(features[, 1], unique = TRUE)
  }
  colnames(mat) <- barcodes
  mat <- as(mat, "CsparseMatrix")
  return(mat)
}

# ========== Custom function to write 10X data ==========
write10x_custom <- function(path, mat) {
  if (!dir.exists(path)) dir.create(path, recursive = TRUE, showWarnings = FALSE)
  
  features_df <- data.frame(
    gene_id = rownames(mat),
    gene_name = rownames(mat),
    feature_type = "Gene Expression",
    stringsAsFactors = FALSE
  )
  
  Matrix::writeMM(mat, file = file.path(path, "matrix.mtx"))
  writeLines(colnames(mat), con = file.path(path, "barcodes.tsv"))
  write.table(features_df, file = file.path(path, "features.tsv"),
              row.names = FALSE, col.names = FALSE, sep = "\t", quote = FALSE)
  
  system(paste("gzip -f", file.path(path, "matrix.mtx")))
  system(paste("gzip -f", file.path(path, "barcodes.tsv")))
  system(paste("gzip -f", file.path(path, "features.tsv")))
  
  cat("  10X format saved to:", path, "\n")
}

# ========== Main workflow ==========
output_dir <- file.path(root_dir, "soupx_corrected")
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

sample_dirs <- list.dirs(root_dir, recursive = FALSE)
sample_dirs <- sample_dirs[grep("^cDNA-", basename(sample_dirs))]
cat("Detected samples:", length(sample_dirs), "\n")
print(basename(sample_dirs))

if (length(sample_dirs) == 0) stop("No samples found")

soupx_results <- list()

for (sample_path in sample_dirs) {
  sample_name <- basename(sample_path)
  cat("\n==========", sample_name, "==========\n")
  
  raw_dir <- file.path(sample_path, "output", "raw_matrix")
  filtered_dir <- file.path(sample_path, "output", "filter_matrix")
  
  if (!dir.exists(raw_dir) || !dir.exists(filtered_dir)) {
    cat("  [Skipped] raw/filter directory not found\n")
    next
  }
  
  tryCatch({
    cat("  Reading raw matrix...\n")
    raw_counts <- read10x_custom(raw_dir)
    cat("  raw:", nrow(raw_counts), "genes x", ncol(raw_counts), "cells\n")
    
    cat("  Reading filtered matrix...\n")
    filtered_counts <- read10x_custom(filtered_dir)
    cat("  filtered:", nrow(filtered_counts), "genes x", ncol(filtered_counts), "cells\n")
    
    # Align gene sets
    cat("  Aligning gene sets...\n")
    common_genes <- intersect(rownames(raw_counts), rownames(filtered_counts))
    cat("  Number of shared genes:", length(common_genes), "\n")
    
    raw_counts <- raw_counts[common_genes, ]
    filtered_counts <- filtered_counts[common_genes, ]
    raw_counts <- as(raw_counts, "CsparseMatrix")
    filtered_counts <- as(filtered_counts, "CsparseMatrix")
    
    cat("  Building SoupChannel...\n")
    sc <- SoupChannel(tod = raw_counts, toc = filtered_counts)
    
    # Seurat clustering
    cat("  Seurat clustering...\n")
    seu <- CreateSeuratObject(counts = filtered_counts)
    seu <- NormalizeData(seu)
    seu <- FindVariableFeatures(seu, selection.method = "vst", nfeatures = 2000)
    seu <- ScaleData(seu)
    seu <- RunPCA(seu, features = VariableFeatures(object = seu))
    seu <- FindNeighbors(seu, dims = 1:20)
    seu <- FindClusters(seu, resolution = 0.8)
    
    # ========== Key fix: setClusters is a function, not a method ==========
    cat("  Passing cluster information...\n")
    sc <- setClusters(sc, setNames(seu$seurat_clusters, colnames(seu)))
    
    # Estimate contamination with SoupX
    cat("  Estimating contamination fraction...\n")
    sc <- autoEstCont(sc, forceAccept = TRUE)
    
    rho <- sc$fit$rhoEst
    cat("  Contamination fraction:", round(rho * 100, 2), "%\n")
    
    corrected_counts <- adjustCounts(sc)
    
    # Save
    sample_out <- file.path(output_dir, sample_name)
    dir.create(sample_out, showWarnings = FALSE, recursive = TRUE)
    
    write10x_custom(file.path(sample_out, "soupx_matrix"), corrected_counts)
    saveRDS(corrected_counts, file = file.path(sample_out, "soupx_counts.rds"))
    
    soupx_results[[sample_name]] <- data.frame(
      sample = sample_name,
      rho = rho,
      rho_percent = round(rho * 100, 2),
      stringsAsFactors = FALSE
    )
    cat("  [Done]\n")
    
  }, error = function(e) {
    cat("  [Error]", conditionMessage(e), "\n")
    soupx_results[[sample_name]] <<- data.frame(
      sample = sample_name, rho = NA, rho_percent = NA,
      error = conditionMessage(e), stringsAsFactors = FALSE
    )
  })
}

cat("\n========== All done ==========\n")
summary_df <- bind_rows(soupx_results)
print(summary_df)
write.csv(summary_df, file.path(output_dir, "soupx_summary.csv"), row.names = FALSE)

In [ ]:
# After SoupX correction

In [ ]:
# ============================================
# Step 2: QC and merging after SoupX correction
# ============================================

suppressMessages(library(tidyverse))
suppressMessages(library(Seurat))
suppressMessages(library(patchwork))
suppressMessages(library(Matrix))
suppressMessages(library(cowplot))
suppressMessages(library(ggplot2))
suppressMessages(library(harmony))

# ---- 1. Path settings ----
soupx_dir <- "./data/soupx_corrected"
save_dir_step2 <- "./data"
fig_dir <- file.path(save_dir_step2, "figures_QC")
dir.create(save_dir_step2, showWarnings = FALSE, recursive = TRUE)
dir.create(fig_dir, showWarnings = FALSE, recursive = TRUE)

# ---- 2. Sample information (donor + region mapping) ----
sample_info <- data.frame(
  sample_label = c("cDNA-250114087", "cDNA-250114088", "cDNA-250110016","cDNA-250110017",
 "cDNA-250110018","cDNA-250110019","cDNA-250114089", "cDNA-250114090",
                   "cDNA-250110006", "cDNA-250110007", "cDNA-250114082", "cDNA-250114083",
                   "cDNA-250114084", "cDNA-250114085", "cDNA-250114086"),
  donor = c(rep("Mq246", 8), rep("Mq277", 7)),
  samples = c(rep("DR-IV-TD", 8), rep("MR-DR-VR-IV", 7)),
  stringsAsFactors = FALSE
)

# ---- 3. Read SoupX-corrected data ----
cat("Reading SoupX-corrected data...\n")
seurat_list <- list()

for (i in 1:nrow(sample_info)) {
  s <- sample_info$sample_label[i]
  cat("  ", s, "\n")
  
  counts <- readRDS(file.path(soupx_dir, s, "soupx_counts.rds"))
  seu <- CreateSeuratObject(counts = counts, project = s)
  
  # Add metadata
  seu$sample_label <- s
  seu$donor <- sample_info$donor[i]
  seu$samples <- sample_info$samples[i]
  
  seurat_list[[s]] <- seu
}

# ---- 4. Merge ----
cat("\nMerging all samples...\n")
seurat <- merge(seurat_list[[1]], y = seurat_list[-1])
cat("After merging:", ncol(seurat), "cells x", nrow(seurat), "genes\n")

# ---- 5. Compute QC metrics ----
seurat$countFeatureRatio <- seurat$nCount_RNA / seurat$nFeature_RNA

MTGenes <- c("ND6","COX1","ND5","ND4","ND2","ND4L","ATP8","CYTB","COX2","ND3","ATP6","ND1","COX3")
found_mt <- MTGenes[MTGenes %in% rownames(seurat)]
cat("Detected MT genes:", length(found_mt), "\n")
seurat[["percent.mt"]] <- PercentageFeatureSet(seurat, features = found_mt)

# ---- 6. Sample filtering (median nFeature > 500) ----
df_meta <- seurat@meta.data
group_med_feature <- df_meta %>% 
  group_by(sample_label) %>% 
  summarise(med = median(nFeature_RNA))
select_group <- group_med_feature[group_med_feature$med > 500, ]$sample_label
seurat <- subset(seurat, sample_label %in% select_group)
cat("Samples after filtering:", length(select_group), "\n")

# ---- 7. Pre-QC statistics (per library) ----
oriCellNum <- ncol(seurat)
cat("Cells before QC:", oriCellNum, "\n")

qc_before <- as.data.frame(table(seurat$sample_label))
colnames(qc_before) <- c("sample_label", "cells_before_QC")

# ---- 8. QC filtering ----
seurat <- subset(seurat,
                 nFeature_RNA > 700 &
                 percent.mt < 5 &
                 countFeatureRatio > 1.2 &
                 nCount_RNA < 100000)
cellNum1 <- ncol(seurat)
cat("Cells after QC:", cellNum1, "(retained", round(cellNum1/oriCellNum*100, 2), "%)\n")

# ---- 8b. Post-QC per-library counts + before/after table ----
qc_after <- as.data.frame(table(seurat$sample_label))
colnames(qc_after) <- c("sample_label", "cells_after_QC")

qc_per_library <- merge(qc_before, qc_after, by = "sample_label", all.x = TRUE)
qc_per_library$cells_after_QC[is.na(qc_per_library$cells_after_QC)] <- 0  # Libraries removed entirely are filled with 0
qc_per_library$retained_pct <- round(
  qc_per_library$cells_after_QC / qc_per_library$cells_before_QC * 100, 1)

# Add donor / region information
qc_per_library <- qc_per_library %>%
  left_join(distinct(sample_info, sample_label, donor, samples), by = "sample_label")

cat("\nCell counts per library before/after QC:\n")
print(qc_per_library, row.names = FALSE)

write.csv(qc_per_library,
          file.path(save_dir_step2, "QC_per_library_cell_counts.csv"),
          row.names = FALSE)
# ---- 9. QC plots ----
Idents(seurat) <- "sample_label"

# Violin plots
p_vln <- VlnPlot(seurat, features = c("nFeature_RNA", "nCount_RNA", "percent.mt"), 
                 pt.size = 0, ncol = 1, group.by = "sample_label")
png(file.path(fig_dir, "QC_violin.png"), width = 1500, height = 1800, res = 150)
print(p_vln)
dev.off()

# Scatter plots
p1 <- FeatureScatter(seurat, "nCount_RNA", "nFeature_RNA", group.by = "sample_label", pt.size = 0.3)
p2 <- FeatureScatter(seurat, "nCount_RNA", "percent.mt", group.by = "sample_label", pt.size = 0.3)
p3 <- FeatureScatter(seurat, "nFeature_RNA", "percent.mt", group.by = "sample_label", pt.size = 0.3)
p_scatter <- (p1 + p2 + p3) + plot_layout(ncol = 3)

png(file.path(fig_dir, "QC_scatter.png"), width = 2000, height = 650, res = 150)
print(p_scatter)
dev.off()

cat("QC plots saved to:", fig_dir, "\n")

# ---- 10. Save Step 2 results ----
filename <- file.path(save_dir_step2, "macaque_DR_soupx_batch1.rds")
saveRDS(seurat, filename)
cat("Step 2 data saved:", filename, "\n")

In [ ]:
# ---- 10. Save Step 2 results ----
filename <- file.path(save_dir_step2, "macaque_DR_soupx_batch1.rds")
saveRDS(seurat, filename)
cat("Step 2 data saved:", filename, "\n")

In [ ]:
# Single-cell analysis

In [ ]:
library(Seurat)
library(dplyr)

seurat <- readRDS("./data/macaque_DR_soupx_batch1.rds")

# Sample filtering (med > 500)
df_meta <- seurat@meta.data
group_med_feature <- df_meta %>% 
  group_by(sample_label) %>% 
  summarise(med = median(nFeature_RNA))
select_group <- group_med_feature[group_med_feature$med > 500, ]$sample_label

seurat <- subset(seurat, sample_label %in% select_group)
oriCellNum <- ncol(seurat)

# QC filtering
seurat$countFeatureRatio <- seurat$nCount_RNA / seurat$nFeature_RNA
MTGenes <- c("ND6","COX3","COX1","ND5","ND4","ND2","ND4L","ATP8","CYTB","COX2","ND3","ATP6","ND1")
seurat[["percent.mt"]] <- PercentageFeatureSet(seurat, features = MTGenes)
seurat_qc <- subset(seurat, nFeature_RNA > 700 & nCount_RNA < 100000 &
                    percent.mt < 5 & countFeatureRatio > 1.2)
cellNum1 <- ncol(seurat_qc)

cat("Cells before QC:", oriCellNum, "\n")
cat("Cells after QC:", cellNum1, "\n")
cat("Removed:", oriCellNum - cellNum1, 
    "(", round((oriCellNum - cellNum1)/oriCellNum*100, 2), "%)\n")

In [ ]:
# Step 2 plots (no downsampling) - revised version
library(Seurat)
library(ggplot2)
library(patchwork)
library(dplyr)

# ==================== Output path settings ====================
save_dir <- "/./data/figures_QC-latest-812"
if (!dir.exists(save_dir)) {
  dir.create(save_dir, recursive = TRUE)
  cat("Created directory:", save_dir, "\n")
}

# ==================== (1) Sample filtering ====================
df_meta <- seurat@meta.data
group_med_feature <- df_meta %>% 
  group_by(sample_label) %>% 
  summarise(med = median(nFeature_RNA))

select_group <- group_med_feature[group_med_feature$med > 500, ]$sample_label
seurat <- subset(seurat, sample_label %in% select_group)

# ==================== (2) Compute QC metrics ====================
seurat$countFeatureRatio <- seurat$nCount_RNA / seurat$nFeature_RNA
MTGenes <- c("ND6","COX3","COX1","ND5","ND4","ND2","ND4L","ATP8","CYTB","COX2","ND3","ATP6","ND1")
seurat[["percent.mt"]] <- PercentageFeatureSet(seurat, features = MTGenes)
seurat <- NormalizeData(seurat)

oriCellNum <- ncol(seurat)

# ==================== Use all data for plotting (no downsampling) ====================
seurat_plot <- seurat
cat("Plotting with ALL", ncol(seurat_plot), "cells\n")

# ==================== New: violin-plot subset (nCount_RNA <= 25000) ====================
seurat_plot_vln <- subset(seurat_plot, nCount_RNA <= 25000)
cat("Violin plot using", ncol(seurat_plot_vln), "cells (nCount_RNA <= 25000)\n")

# ==================== Publication-grade violin plot function (colored by donor, nFeature ticks every 1000) ====================
myVlnPlot <- function(seurat_obj, feature, group.by = "sample_label", 
                      fill.by = "donor", title = NULL, ylabel = NULL) {
  meta <- seurat_obj@meta.data
  meta$group <- factor(meta[[group.by]])
  meta$fill_group <- factor(meta[[fill.by]])
  
  # Donor colors: Mq246 blue, Mq277 orange (adjust to actual donor names if needed)
  donor_colors <- c("Mq246" = "#1f77b4", "Mq277" = "#ff7f0e")
  
  p <- ggplot(meta, aes(x = group, y = .data[[feature]], fill = fill_group)) +
    geom_violin(scale = "width", trim = TRUE, linewidth = 0.3) +
    geom_violin(scale = "width", trim = TRUE, linewidth = 0.3, alpha = 0.85) +
    scale_fill_manual(values = donor_colors, name = "Donor") +
    theme_classic() +
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1, size = 10, color = "black"),
      axis.text.y = element_text(size = 10, color = "black"),
      axis.title.x = element_blank(),
      axis.title.y = element_text(size = 12, face = "bold"),
      plot.title = element_text(hjust = 0.5, face = "bold", size = 13),
      legend.position = "right",
      legend.key.size = unit(0.4, "cm"),
      legend.text = element_text(size = 9),
      legend.title = element_text(size = 10, face = "bold"),
      panel.border = element_rect(color = "black", fill = NA, linewidth = 0.5)
    ) +
    labs(title = title, y = ifelse(is.null(ylabel), feature, ylabel))
  
  # nFeature_RNA tick marks every 1000
  if (feature == "nFeature_RNA") {
    ymax <- max(meta[[feature]], na.rm = TRUE)
    p <- p + scale_y_continuous(breaks = seq(0, ceiling(ymax / 1000) * 1000, 1000))
  }
  
  return(p)
}

# ==================== (a) Pre-QC violin plots (nCount_RNA <= 25000 only, colored by donor) ====================
cat("\n--- Saving (a) Pre-QC Violin ---\n")
p_a1 <- myVlnPlot(seurat_plot_vln, "nFeature_RNA", title = "Gene Count", ylabel = "nFeature_RNA")
p_a2 <- myVlnPlot(seurat_plot_vln, "nCount_RNA", title = "UMI Count", ylabel = "nCount_RNA")
p_a3 <- myVlnPlot(seurat_plot_vln, "percent.mt", title = "Mitochondrial %", ylabel = "percent.mt")

vln_pre <- (p_a1 + p_a2 + p_a3) + 
  plot_layout(ncol = 3, widths = c(1, 1, 1)) +
  plot_annotation(title = "Pre-QC Metrics by Sample (Colored by Donor)", 
                    theme = theme(plot.title = element_text(size = 14, face = "bold", hjust = 0.5)))

png(file.path(save_dir, "a_violin_preQC_byDonor.png"), width = 2000, height = 650, res = 150)
print(vln_pre)
dev.off()
cat("Saved: a_violin_preQC_byDonor.png\n")

# ==================== (b) Pre-QC scatter plots (no downsampling, keep all data) ====================
cat("\n--- Saving (b) Pre-QC Scatter ---\n")

scatter1 <- FeatureScatter(
  seurat_plot, feature1 = "nCount_RNA", feature2 = "nFeature_RNA",
  group.by = "sample_label", pt.size = 0.3
) + ggtitle("UMI vs Gene Count") + 
  theme_classic() +
  theme(legend.position = "none", plot.title = element_text(size = 12, face = "bold"))

scatter2 <- FeatureScatter(
  seurat_plot, feature1 = "nCount_RNA", feature2 = "percent.mt",
  group.by = "sample_label", pt.size = 0.3
) + ggtitle("UMI vs MT%") + 
  theme_classic() +
  theme(legend.position = "none", plot.title = element_text(size = 12, face = "bold"))

scatter3 <- FeatureScatter(
  seurat_plot, feature1 = "nFeature_RNA", feature2 = "percent.mt",
  group.by = "sample_label", pt.size = 0.3
) + ggtitle("Gene Count vs MT%") + 
  theme_classic() +
  theme(legend.position = "right", plot.title = element_text(size = 12, face = "bold"),
        legend.key.size = unit(0.3, "cm"), legend.text = element_text(size = 8))

scatter_pre <- (scatter1 + scatter2 + scatter3) + 
  plot_layout(ncol = 3, widths = c(1, 1, 1.2)) +
  plot_annotation(title = "Pre-QC Scatter Plots",
                    theme = theme(plot.title = element_text(size = 14, face = "bold", hjust = 0.5)))

png(file.path(save_dir, "b_scatter_preQC.png"), width = 2000, height = 650, res = 150)
print(scatter_pre)
dev.off()
ggsave(file.path(save_dir, "b_scatter_preQC.pdf"),
       scatter_pre, width = 13.33, height = 4.33, device = cairo_pdf)   # Newly added
cat("Saved: b_scatter_preQC.png / .pdf\n")

# ==================== (c) QC filtering ====================
seurat_qc <- subset(seurat, 
                    nFeature_RNA > 700 & 
                    nCount_RNA < 100000 &
                    percent.mt < 5 & 
                    countFeatureRatio > 1.2)

cellNum1 <- ncol(seurat_qc)
cat("\nPost-QC:", cellNum1, "cells (removed", oriCellNum - cellNum1, 
    round((oriCellNum - cellNum1)/oriCellNum * 100, 1), "%)\n")

# ==================== New: post-QC violin-plot subset (nCount_RNA <= 25000) ====================
seurat_qc_vln <- subset(seurat_qc, nCount_RNA <= 25000)
cat("Post-QC Violin plot using", ncol(seurat_qc_vln), "cells (nCount_RNA <= 25000)\n")

# ==================== (d) Post-QC violin plots (nCount_RNA <= 25000 only, colored by donor) ====================
cat("\n--- Saving (c) Post-QC Violin ---\n")

p_d1 <- myVlnPlot(seurat_qc_vln, "nFeature_RNA", title = "Gene Count", ylabel = "nFeature_RNA")
p_d2 <- myVlnPlot(seurat_qc_vln, "nCount_RNA", title = "UMI Count", ylabel = "nCount_RNA")
p_d3 <- myVlnPlot(seurat_qc_vln, "percent.mt", title = "Mitochondrial %", ylabel = "percent.mt")

vln_post <- (p_d1 + p_d2 + p_d3) + 
  plot_layout(ncol = 3, widths = c(1, 1, 1)) +
  plot_annotation(title = "Post-QC Metrics by Sample (Colored by Donor)",
                    theme = theme(plot.title = element_text(size = 14, face = "bold", hjust = 0.5)))

png(file.path(save_dir, "c_violin_postQC_byDonor.png"), width = 2000, height = 650, res = 150)
print(vln_post)
dev.off()
ggsave(file.path(save_dir, "c_violin_postQC_byDonor.pdf"),
       vln_post, width = 13.33, height = 4.33, device = cairo_pdf)   # Newly added
cat("Saved: c_violin_postQC_byDonor.png / .pdf\n")
# ==================== Verify output files ====================
cat("\n========== Generated Files ==========\n")
files <- c("a_violin_preQC_byDonor.png", "b_scatter_preQC.png", 
           "c_violin_postQC_byDonor.png")
for (f in files) {
  fp <- file.path(save_dir, f)
  if (file.exists(fp)) {
    cat("OK:", f, "-", round(file.size(fp)/1024, 1), "KB\n")
  } else {
    cat("MISSING:", f, "\n")
  }
}

cat("\n========== QC Summary ==========\n")
cat("Original cells:", oriCellNum, "\n")
cat("Post-QC cells:", cellNum1, "\n")
cat("Removed:", oriCellNum - cellNum1, 
    "(", round((oriCellNum - cellNum1)/oriCellNum * 100, 2), "%)\n")
cat("Save directory:", save_dir, "\n")
cat("==================================\n")

In [ ]:
library(ggplot2)

# Extract metadata
meta <- seurat@meta.data

# Scatter plot: nCount_RNA vs nFeature_RNA, colored by percent.mt
p <- ggplot(meta, aes(x = nCount_RNA, y = nFeature_RNA, color = percent.mt)) +
  geom_point(size = 0.3, alpha = 0.5) +
  scale_color_distiller(palette = "Reds", direction = 1, name = "percent.mt (%)") +
  labs(x = "nCount_RNA", y = "nFeature_RNA") +
  theme_classic() +
  theme(
    axis.text = element_text(size = 11, color = "black"),
    axis.title = element_text(size = 12, face = "bold"),
    legend.title = element_text(size = 10),
    legend.text = element_text(size = 9)
  )

# Save + display
fig_dir <- "./data/figures_QC-latest-812"
dir.create(fig_dir, showWarnings = FALSE, recursive = TRUE)

png(file.path(fig_dir, "scatter_nCount_nFeature_mt.png"), width = 800, height = 700, res = 150)
print(p)
dev.off()

cat("Saved:", file.path(fig_dir, "scatter_nCount_nFeature_mt.png"), "\n")

In [ ]:
library(Seurat)
library(Matrix)
library(dplyr)
library(ggplot2)
library(patchwork)

# ========== Paths ==========
soupx_dir <- "./data/soupx_corrected"
save_dir_step2 <- "./data"
fig_dir <- file.path(save_dir_step2, "figures_QC-latest-812")
dir.create(fig_dir, showWarnings = FALSE, recursive = TRUE)

# ========== Sample information ==========
sample_info <- data.frame(
  sample_label = c("cDNA-250114087", "cDNA-250114088", "cDNA-250110016","cDNA-250110017",
 "cDNA-250110018","cDNA-250110019","cDNA-250114089", "cDNA-250114090",
                   "cDNA-250110006", "cDNA-250110007", "cDNA-250114082", "cDNA-250114083",
                   "cDNA-250114084", "cDNA-250114085", "cDNA-250114086"),
  donor = c(rep("Mq246", 8), rep("Mq277", 7)),
  samples = c(rep("DR-IV-TD", 8), rep("MR-DR-VR-IV", 7)),
  stringsAsFactors = FALSE
)

# ========== 1. Read all SoupX counts ==========
cat("Reading SoupX-corrected data...\n")
counts_list <- list()

for (i in 1:nrow(sample_info)) {
  s <- sample_info$sample_label[i]
  cat("  ", s, "\n")
  
  counts <- readRDS(file.path(soupx_dir, s, "soupx_counts.rds"))
  
  # Add sample prefix to barcodes to avoid duplicates across samples
  colnames(counts) <- paste0(colnames(counts), "-", s)
  
  counts_list[[s]] <- counts
}

# ========== 2. Take the union of all genes and align each matrix (fill missing with 0) ==========
cat("\nAligning genes...\n")
all_genes <- unique(unlist(lapply(counts_list, rownames)))
cat("Union of genes across all samples:", length(all_genes), "\n")

counts_list_aligned <- lapply(names(counts_list), function(s) {
  mat <- counts_list[[s]]
  missing_genes <- setdiff(all_genes, rownames(mat))
  
  if (length(missing_genes) > 0) {
    zero_mat <- Matrix(0, nrow = length(missing_genes), ncol = ncol(mat),
                       dimnames = list(missing_genes, colnames(mat)))
    mat <- rbind(mat, zero_mat)
  }
  # Uniform ordering
  mat <- mat[all_genes, , drop = FALSE]
  return(mat)
})
names(counts_list_aligned) <- names(counts_list)

# ========== 3. cbind-merge all counts (single layer) ==========
cat("Merging counts...\n")
combined_counts <- do.call(cbind, counts_list_aligned)
cat("After merging:", nrow(combined_counts), "genes x", ncol(combined_counts), "cells\n")
# Expected: 21883 genes x 137092 cells (before QC)

# ========== 4. Create Seurat object (single counts layer) ==========
seurat <- CreateSeuratObject(counts = combined_counts)
cat("Seurat object:", nrow(seurat), "features,", ncol(seurat), "cells\n")
cat("Layers:", names(seurat@assays$RNA@layers), "\n")

# ========== 5. Add metadata ==========
cell_counts <- sapply(counts_list_aligned, ncol)
seurat$sample_label <- rep(sample_info$sample_label, cell_counts)
seurat$donor <- rep(sample_info$donor, cell_counts)
seurat$samples <- rep(sample_info$samples, cell_counts)

# ========== 6. QC computation ==========
seurat$countFeatureRatio <- seurat$nCount_RNA / seurat$nFeature_RNA

MTGenes <- c("ND6","COX3","COX1","ND5","ND4","ND2","ND4L","ATP8","CYTB","COX2","ND3","ATP6","ND1")
found_mt <- MTGenes[MTGenes %in% rownames(seurat)]
seurat[["percent.mt"]] <- PercentageFeatureSet(seurat, features = found_mt)

# ========== 7. Sample filtering (median nFeature > 500) ==========
df_meta <- seurat@meta.data
group_med_feature <- df_meta %>% 
  group_by(sample_label) %>% 
  summarise(med = median(nFeature_RNA))
select_group <- group_med_feature[group_med_feature$med > 500, ]$sample_label
seurat <- subset(seurat, sample_label %in% select_group)

oriCellNum <- ncol(seurat)
cat("Before QC:", oriCellNum, "cells\n")

# ========== 8. Pre-QC plots ==========
# (Original plotting code goes here, omitted)

# ========== 9. QC filtering ==========
seurat_qc <- subset(seurat,
                    nFeature_RNA > 700 &
                    percent.mt < 5 &
                    countFeatureRatio > 1.2 &
                    nCount_RNA < 100000)

cat("After QC:", ncol(seurat_qc), "cells (retained", round(ncol(seurat_qc)/oriCellNum*100,2), "%)\n")
# Expected: 135419

# ========== 10. Save RDS (critical!) ==========
seurat <- seurat_qc
rds_file <- file.path(save_dir_step2, "macaque_DR_soupx_batch1.rds")
saveRDS(seurat, rds_file)
cat("RDS saved:", rds_file, "\n")
cat("Final:", ncol(seurat), "cells x", nrow(seurat), "genes, 1 layer\n")

In [ ]:
readDir <- "./data/"
seurat1 <- readRDS(paste0(readDir, "macaque_DR_soupx_batch1.rds"))

In [ ]:
# Assess doublet rate

In [ ]:
library(Seurat)
library(scDblFinder)
library(SingleCellExperiment)
library(dplyr)
library(ggplot2)

seurat1$sample_label <- as.character(seurat1$sample_label)
samples <- unique(seurat1$sample_label)

# Initialize (explicitly set as character to avoid factor issues)
seurat1$scDblFinder.class <- NA_character_
seurat1$scDblFinder.score <- NA_real_

for (s in samples) {
  message("Processing sample: ", s)
  
  subset_obj <- subset(seurat1, subset = sample_label == s)
  if (ncol(subset_obj) < 100) next
  
  # Normalization + PCA
  subset_obj <- NormalizeData(subset_obj)
  subset_obj <- FindVariableFeatures(subset_obj, selection.method = "vst", nfeatures = 2000)
  subset_obj <- ScaleData(subset_obj)
  subset_obj <- RunPCA(subset_obj, features = VariableFeatures(object = subset_obj))
  
  # Convert to SCE and run scDblFinder
  sce <- as.SingleCellExperiment(subset_obj)
  sce <- scDblFinder(sce)
  
  # ========== Key fix: write results back with AddMetaData matched by cell name ==========
  meta_to_add <- data.frame(
    scDblFinder.class = as.character(sce$scDblFinder.class),
    scDblFinder.score = sce$scDblFinder.score,
    stringsAsFactors = FALSE
  )
  rownames(meta_to_add) <- colnames(sce)  # Ensure cell names are aligned
  
  seurat1 <- AddMetaData(seurat1, metadata = meta_to_add)
  
  dbl_n <- sum(sce$scDblFinder.class == "doublet", na.rm = TRUE)
  message("  Done. Doublets: ", dbl_n, "/", ncol(subset_obj), 
          " (", round(dbl_n / ncol(subset_obj) * 100, 2), "%)")
}

# ========== Doublet rate per sample ==========
doublet_summary <- seurat1@meta.data %>%
  group_by(sample_label) %>%
  summarise(
    total_cells = n(),
    doublet_cells = sum(scDblFinder.class == "doublet", na.rm = TRUE),
    doublet_rate = round(doublet_cells / total_cells * 100, 2)
  ) %>%
  arrange(desc(doublet_rate))

print(doublet_summary)

# ========== Visualization ==========
# Bar plot
p1 <- ggplot(doublet_summary, aes(x = reorder(sample_label, -doublet_rate), y = doublet_rate)) +
  geom_bar(stat = "identity", fill = "steelblue") +
  geom_text(aes(label = paste0(doublet_rate, "%")), vjust = -0.5, size = 3) +
  labs(x = "Sample", y = "Doublet Rate (%)", title = "scDblFinder Doublet Rate per Sample") +
  theme_bw(base_size = 12) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))
print(p1)

# If UMAP exists, show doublet distribution
if ("umap" %in% names(seurat1@reductions)) {
  p2 <- DimPlot(seurat1, group.by = "scDblFinder.class", label = FALSE,
                cols = c("singlet" = "grey", "doublet" = "red")) +
    ggtitle("scDblFinder Predictions")
  print(p2)
}

# ========== Filter doublets and save ==========
seurat1_clean <- subset(seurat1, subset = scDblFinder.class == "singlet")

message("Before filtering: ", ncol(seurat1), " cells")
message("After filtering:  ", ncol(seurat1_clean), " cells")
message("Removed:          ", ncol(seurat1) - ncol(seurat1_clean), " doublets")

# Save
saveRDS(seurat1_clean, paste0(readDir, "macaque_DR_soupx_batch1_dblCleaned.rds"))

In [ ]:
# 1. View all metadata column names
colnames(seurat1@meta.data)

# 2. View sample_label distribution
table(seurat1@meta.data$sample_label)

# 3. View samples distribution
table(seurat1@meta.data$samples)

# 4. View cell barcode format
head(Cells(seurat1))

# 5. Extract suffix from barcode as library ID
library(dplyr)
seurat1$library_id <- sapply(strsplit(Cells(seurat1), "-"), function(x) tail(x, 1))
table(seurat1$library_id)





In [ ]:
# === First read the RDS file saved in the previous step ===
readDir <- "./data/"

# If you saved this filename in the previous step:
seurat <- readRDS(paste0(readDir, "macaque_DR_soupx_batch1_dblCleaned.rds"))

# If the file above does not exist, try the post-doublet but pre-QC version:
# seurat <- readRDS(paste0(readDir, "macaque_DR_soupx_batch1_dblCleaned.rds"))

cat("Loaded object with", ncol(seurat), "cells\n")

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(patchwork)

# ============================================
# 0. Set paths & load data
# ============================================
readDir  <- "./data/"
save_dir <- "./data"

dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)   # Added here

after_doublet   <- ncol(seurat)   
# ============================================
# 1. Handle donor column
# ============================================
if (!"donor" %in% colnames(seurat@meta.data)) {
  seurat$donor <- seurat$sample_label
  donor_colors <- NULL
  message("Note: No 'donor' column found. Using sample_label for coloring.")
} else {
  donor_colors <- c("Mq246" = "#1f77b4", "Mq277" = "#ff7f0e")
}

# ============================================
# 2. Sample filtering (median nFeature > 500)
# ============================================
df_meta <- seurat@meta.data
group_med_feature <- df_meta %>% 
  group_by(sample_label) %>% 
  summarise(med = median(nFeature_RNA))

select_group <- group_med_feature[group_med_feature$med > 500, ]$sample_label
seurat <- subset(seurat, sample_label %in% select_group)

# ============================================
# 3. Compute QC metrics
# ============================================
seurat$countFeatureRatio <- seurat$nCount_RNA / seurat$nFeature_RNA

MTGenes <- c("ND6","COX3","COX1","ND5","ND4","ND2","ND4L","ATP8","CYTB","COX2","ND3","ATP6","ND1")
seurat[["percent.mt"]] <- PercentageFeatureSet(seurat, features = MTGenes)
seurat <- NormalizeData(seurat)

oriCellNum <- ncol(seurat)

# ============================================
# 4. Prepare plotting data
# ============================================
seurat_plot <- seurat
cat("Plotting with ALL", ncol(seurat_plot), "cells\n")

seurat_plot_vln <- subset(seurat_plot, nCount_RNA <= 25000)
cat("Violin plot using", ncol(seurat_plot_vln), "cells (nCount_RNA <= 25000)\n")

# ============================================
# 5. Violin plot function
# ============================================
myVlnPlot <- function(seurat_obj, feature, group.by = "sample_label", 
                      fill.by = "donor", title = NULL, ylabel = NULL) {
  meta <- seurat_obj@meta.data
  meta$group <- factor(meta[[group.by]])
  meta$fill_group <- factor(meta[[fill.by]])
  
  p <- ggplot(meta, aes(x = group, y = .data[[feature]], fill = fill_group)) +
    geom_violin(scale = "width", trim = TRUE, linewidth = 0.3, alpha = 0.85)
  
  if (!is.null(donor_colors)) {
    p <- p + scale_fill_manual(values = donor_colors, name = "Donor")
  } else {
    p <- p + scale_fill_discrete(name = fill.by)
  }
  
  p <- p + theme_classic() +
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1, size = 10, color = "black"),
      axis.text.y = element_text(size = 10, color = "black"),
      axis.title.x = element_blank(),
      axis.title.y = element_text(size = 12, face = "bold"),
      plot.title = element_text(hjust = 0.5, face = "bold", size = 13),
      legend.position = "right",
      legend.key.size = unit(0.4, "cm"),
      legend.text = element_text(size = 9),
      legend.title = element_text(size = 10, face = "bold"),
      panel.border = element_rect(color = "black", fill = NA, linewidth = 0.5)
    ) +
    labs(title = title, y = ifelse(is.null(ylabel), feature, ylabel))
  
  if (feature == "nFeature_RNA") {
    ymax <- max(meta[[feature]], na.rm = TRUE)
    p <- p + scale_y_continuous(breaks = seq(0, ceiling(ymax / 1000) * 1000, 1000))
  }
  
  return(p)
}

# ============================================
# 6. (a) Post-doublet QC violin plots
# ============================================
cat("\n--- Saving (a) Post-Doublet Violin ---\n")
p_a1 <- myVlnPlot(seurat_plot_vln, "nFeature_RNA", title = "Gene Count", ylabel = "nFeature_RNA")
p_a2 <- myVlnPlot(seurat_plot_vln, "nCount_RNA", title = "UMI Count", ylabel = "nCount_RNA")
p_a3 <- myVlnPlot(seurat_plot_vln, "percent.mt", title = "Mitochondrial %", ylabel = "percent.mt")

vln_pre <- (p_a1 + p_a2 + p_a3) + 
  plot_layout(ncol = 3, widths = c(1, 1, 1)) +
  plot_annotation(title = "Post-Doublet Removal QC Metrics", 
                    theme = theme(plot.title = element_text(size = 14, face = "bold", hjust = 0.5)))

png(file.path(save_dir, "a_violin_postDoublet_byDonor.png"), width = 2000, height = 650, res = 150)
print(vln_pre)
dev.off()
cat("Saved: a_violin_postDoublet_byDonor.png\n")

# ============================================
# 7. (b) Scatter plots
# ============================================
cat("\n--- Saving (b) Post-Doublet Scatter ---\n")

scatter1 <- FeatureScatter(
  seurat_plot, feature1 = "nCount_RNA", feature2 = "nFeature_RNA",
  group.by = "sample_label", pt.size = 0.3
) + ggtitle("UMI vs Gene Count") + 
  theme_classic() +
  theme(legend.position = "none", plot.title = element_text(size = 12, face = "bold"))

scatter2 <- FeatureScatter(
  seurat_plot, feature1 = "nCount_RNA", feature2 = "percent.mt",
  group.by = "sample_label", pt.size = 0.3
) + ggtitle("UMI vs MT%") + 
  theme_classic() +
  theme(legend.position = "none", plot.title = element_text(size = 12, face = "bold"))

scatter3 <- FeatureScatter(
  seurat_plot, feature1 = "nFeature_RNA", feature2 = "percent.mt",
  group.by = "sample_label", pt.size = 0.3
) + ggtitle("Gene Count vs MT%") + 
  theme_classic() +
  theme(legend.position = "right", plot.title = element_text(size = 12, face = "bold"),
        legend.key.size = unit(0.3, "cm"), legend.text = element_text(size = 8))

scatter_pre <- (scatter1 + scatter2 + scatter3) + 
  plot_layout(ncol = 3, widths = c(1, 1, 1.2)) +
  plot_annotation(title = "Post-Doublet Removal Scatter Plots",
                    theme = theme(plot.title = element_text(size = 14, face = "bold", hjust = 0.5)))

png(file.path(save_dir, "b_scatter_postDoublet.png"), width = 2000, height = 650, res = 150)
print(scatter_pre)
dev.off()
cat("Saved: b_scatter_postDoublet.png\n")

# ============================================
# 8. (c) QC filtering
# ============================================
seurat_qc <- subset(seurat, 
                    nFeature_RNA > 700 & 
                    nCount_RNA < 100000 &
                    percent.mt < 5 & 
                    countFeatureRatio > 1.2)

cellNum1 <- ncol(seurat_qc)
cat("\nPost-QC:", cellNum1, "cells (removed", oriCellNum - cellNum1, 
    round((oriCellNum - cellNum1)/oriCellNum * 100, 1), "%)\n")

seurat_qc_vln <- subset(seurat_qc, nCount_RNA <= 25000)
cat("Post-QC Violin plot using", ncol(seurat_qc_vln), "cells (nCount_RNA <= 25000)\n")

# ============================================
# 9. (d) Post-QC violin plots
# ============================================
cat("\n--- Saving (c) Post-QC Violin ---\n")

p_d1 <- myVlnPlot(seurat_qc_vln, "nFeature_RNA", title = "Gene Count", ylabel = "nFeature_RNA")
p_d2 <- myVlnPlot(seurat_qc_vln, "nCount_RNA", title = "UMI Count", ylabel = "nCount_RNA")
p_d3 <- myVlnPlot(seurat_qc_vln, "percent.mt", title = "Mitochondrial %", ylabel = "percent.mt")

vln_post <- (p_d1 + p_d2 + p_d3) + 
  plot_layout(ncol = 3, widths = c(1, 1, 1)) +
  plot_annotation(title = "Post-Doublet Removal + Post-QC Metrics",
                    theme = theme(plot.title = element_text(size = 14, face = "bold", hjust = 0.5)))

png(file.path(save_dir, "c_violin_postQC_postDoublet.png"), width = 2000, height = 650, res = 150)
print(vln_post)
dev.off()
cat("Saved: c_violin_postQC_postDoublet.png\n")

# ============================================
# 10. Save final object
# ============================================
saveRDS(seurat_qc, paste0(readDir, "macaque_DR_soupx_batch1_dblCleaned_QCed.rds"))

# ============================================
# 11. Summary
# ============================================
cat("\n========== Summary ==========\n")
cat("Original cells:        ", original_total, "\n")
cat("After doublet removal: ", after_doublet, "\n")
cat("After sample filter:   ", oriCellNum, "\n")
cat("After QC filter:       ", cellNum1, "\n")
cat("Total removed:         ", original_total - cellNum1, 
    "(", round((original_total - cellNum1)/original_total * 100, 2), "%)\n")
cat("Save directory:        ", save_dir, "\n")
cat("Final RDS:             ", paste0(readDir, "macaque_DR_soupx_batch1_dblCleaned_QCed.rds"), "\n")
cat("==================================\n")

In [ ]:
# Ensure percent.mt has been computed
if (!"percent.mt" %in% colnames(seurat@meta.data)) {
  MTGenes <- c("ND6","COX3","COX1","ND5","ND4","ND2","ND4L","ATP8","CYTB","COX2","ND3","ATP6","ND1")
  seurat[["percent.mt"]] <- PercentageFeatureSet(seurat, features = MTGenes)
}

meta <- seurat@meta.data

p <- ggplot(meta, aes(x = nCount_RNA, y = nFeature_RNA, color = percent.mt)) +
  geom_point(size = 0.3, alpha = 0.5) +
  scale_color_distiller(palette = "Reds", direction = 1, name = "percent.mt (%)") +
  labs(x = "nCount_RNA", y = "nFeature_RNA") +
  theme_classic() +
  theme(
    axis.text = element_text(size = 11, color = "black"),
    axis.title = element_text(size = 12, face = "bold"),
    legend.title = element_text(size = 10),
    legend.text = element_text(size = 9)
  )

fig_dir <- "./data"
dir.create(fig_dir, showWarnings = FALSE, recursive = TRUE)

png(file.path(fig_dir, "scatter_nCount_nFeature_mt.png"), width = 800, height = 700, res = 150)
print(p)
dev.off()

cat("Saved:", file.path(fig_dir, "scatter_nCount_nFeature_mt.png"), "\n")

In [ ]:
# Step 3: annotate major classes + random forest

In [ ]:
# Plot each clusters
myHighlightDimPlot = function(seuratObj, highlightBy, size=1, alpha=1, showAllPoints=FALSE, ...) {
    umapTb = as_tibble(seuratObj@reductions$umap@cell.embeddings)
    if (!identical(rownames(seuratObj@reductions$umap@cell.embeddings), rownames(seuratObj@meta.data)))
        stop("Error: cell names not identical!")
    umapTb = cbind(umapTb, seuratObj@meta.data)
    highlightByClass = sort(unique(umapTb[[highlightBy]]))
    gpList = lapply(highlightByClass, function(cl) {
        subUmapTb = subset(umapTb, umapTb[[highlightBy]] == cl)
        if (showAllPoints) {
            gp = ggplot() + 
                geom_point(data=umapTb, aes(x=UMAP_1, y=UMAP_2), color="grey", shape=16, size=size, alpha=alpha) +
                geom_point(data=subUmapTb, aes(x=UMAP_1, y=UMAP_2), color="red", shape=16, size=size, alpha=alpha) +
                xlim(min(umapTb$UMAP_1), max(umapTb$UMAP_1)) +
                ylim(min(umapTb$UMAP_2), max(umapTb$UMAP_2)) +
                coord_fixed() +
                theme_void() 
        } else {
            gp = ggplot() + 
                geom_point(data=umapTb, aes(x=UMAP_1, y=UMAP_2), color="grey", shape=16, size=size, alpha=alpha) +
                geom_point(data=subUmapTb, aes(x=UMAP_1, y=UMAP_2), color="red", shape=16, size=size, alpha=alpha) +
                xlim(min(umapTb$UMAP_1), max(umapTb$UMAP_1)) +
                ylim(min(umapTb$UMAP_2), max(umapTb$UMAP_2)) +
                coord_fixed() +
                theme_void()
        }
        return(gp)
    })
    gpAll = plot_grid(plotlist=gpList, labels=highlightByClass, ...)
    return(gpAll)
}

In [ ]:
# # 1. Update Rcpp first
# install.packages("Rcpp")

# # 2. Then fully restart R (required!)
# # RStudio: Session -> Restart R
# # Or Jupyter: Kernel -> Restart

In [ ]:
suppressMessages(library(tidyverse))
suppressMessages(library(parallel))
suppressMessages(library(Seurat))
suppressMessages(library(harmony))
suppressMessages(library(SeuratData))
suppressMessages(library(patchwork))
suppressMessages(library(Matrix))
suppressMessages(library(viridis))
suppressMessages(library(cowplot))
suppressMessages(library(ggsci))
suppressMessages(library(pheatmap))
suppressMessages(library(RColorBrewer))
suppressMessages(library(reshape2))
suppressMessages(library(LSD))
suppressMessages(library(readxl))
suppressMessages(library(ggrepel))
suppressMessages(library(randomForest))


In [ ]:
plan("multicore", workers = 12)
options(future.globals.maxSize = 500000 * 1024^5)

In [ ]:
readDir <- "./data/"
seurat1 <- readRDS(paste0(readDir, "macaque_DR_soupx_batch1_dblCleaned_QCed.rds"))

In [ ]:
saveDir <- "./data"
dir.create(saveDir, showWarnings = FALSE, recursive = TRUE)

In [ ]:
classFiltSeurat=seurat1
classFiltSeurat

In [ ]:
classFiltSeurat = SCTransform(
classFiltSeurat, assay="RNA",
ncells=ncol(classFiltSeurat[["RNA"]]),
variable.features.n=5000,
return.only.var.genes = T,
vars.to.regress="percent.mt",
method="glmGamPoi"
)

In [ ]:
classFiltSeurat = classFiltSeurat %>%
RunPCA(npcs=100, verbose = FALSE) %>%
FindNeighbors(dims = 1:50) %>%
FindClusters(verbose = FALSE) %>%
RunUMAP(dims = 1:50)
saveRDS(classFiltSeurat,'./data/20260821batch_merge_batchharmony_pre.rds')

In [ ]:
classFiltSeurat = readRDS('./data/20260821batch_merge_batchharmony_pre.rds')
# Run harmony
DefaultAssay(classFiltSeurat) = "SCT"
options(repr.plot.width=6, repr.plot.height=2.5)

In [ ]:
classFiltSeurat@meta.data$donor = gsub('Q','q',classFiltSeurat@meta.data$donor)

In [ ]:
classFiltSeurat@meta.data

In [ ]:
classFiltSeurat = RunHarmony(
classFiltSeurat, group.by.vars="donor", plot_convergence=TRUE,
assay.use="SCT", reduction.use = "pca", dims.use=1:30
)

In [ ]:
classFiltSeurat <- classFiltSeurat %>%
  FindNeighbors(reduction = "harmony", dims = 1:30, k.param = 20) %>%   # 20 is sufficient
  FindClusters(verbose = FALSE) %>%
  RunUMAP(reduction = "harmony", dims = 1:30, n.neighbors = 20)
saveRDS(classFiltSeurat,'./data/20260821harmonydonorbatch_merge_batchharmony.rds')

In [ ]:
seurat=readRDS('./data/20260821harmonydonorbatch_merge_batchharmony.rds')

In [ ]:
# seurat@meta.data
seurat@meta.data$donor = gsub('Q','q',seurat@meta.data$donor)
seurat@meta.data

In [ ]:
# Custom plotting function
myHighlightDimPlot <- function(seurat, highlightBy, showAllPoints = TRUE, size = 0.1) {
  # Extract UMAP data
  umap_data <- as.data.frame(seurat@reductions$umap@cell.embeddings)
  umap_data$group <- seurat@meta.data[[highlightBy]]
  
  # Check and adjust column names
  colnames(umap_data)[1:2] <- c("UMAP_1", "UMAP_2")  # Ensure column names are correct
  
  # Get unique donor list
  donors <- unique(umap_data$group)
  
  # Create list to store plots
  plot_list <- list()
  
  # Loop over each donor and plot separately
  for (donor in donors) {
    # Create base plot
    p <- ggplot(umap_data, aes(x = UMAP_1, y = UMAP_2)) +
      theme_minimal() +
      theme(legend.position = "right")
    
    # Show all points if required
    if (showAllPoints) {
      p <- p + geom_point(data = umap_data, aes(x = UMAP_1, y = UMAP_2), color = "gray", size = size)
    }
    
    # Highlight current donor
    p <- p + geom_point(data = subset(umap_data, group == donor), aes(x = UMAP_1, y = UMAP_2, color = group), size = size)
    
    # Add title
    p <- p + ggtitle(paste("Donor:", donor))
    
    # Add plot to list
    plot_list[[donor]] <- p
  }
  
  # Combine plots side by side with patchwork
  combined_plot <- wrap_plots(plot_list, ncol = 2)
  
  return(combined_plot)
}

In [ ]:
options(repr.plot.width=14, repr.plot.height=7)
myHighlightDimPlot(seurat, highlightBy="donor", showAllPoints=T, size=0.1)

In [ ]:
# Define selectMarkers vector
selectMarkers = c(
    'SLC17A6',# Excit
    "GAD1", "GAD2", # Inhib,
    "GFAP",'EDNRB', # Astrocytes
    "FOXJ1",  "RSPH1",# Ependymal
    "TAGLN", "ALPHA-SMA", # VSMC
    'COL1A1','LUM',# VLMC
    'FLT1','RAMP2',# Endothelial
    "MCAM", "RGS5", # Pericytes
    "VCAN",'SOX6', # OPC
    "PLP1",'MOBP', # Oligo
    'C1QB','OLR1',# Microglia
    'Olig2',#Polydendrocytes
    'CD52',#Lymphocytes
    'MRC1',#pvMFs
    'TAGLN',#Smooth Muscle
    'TPH2','SLC6A4',#Serotonergic Neurons
    'DRD2','TH',#Dopaminergic Neurons
    'CCK','CARTPT',#Peptidergic Neurons
    'PPP1R1B','BCL11B','PDE1B',  
    'SLC5A7','CHAT',#choli
    "SNAP25"#Neuron
)

# Remove empty elements
selectMarkers = selectMarkers[!is.null(selectMarkers) & selectMarkers != ""]

# Deduplicate
selectMarkers = selectMarkers[!duplicated(selectMarkers)]

# Set Seurat identities
Idents(seurat) = "seurat_clusters"

# Set plotting parameters
options(repr.plot.height=10, repr.plot.width=60)

# Draw dot plot
DotPlot(seurat, features = selectMarkers)

In [ ]:
seurat2=seurat 

In [ ]:
seurat2@meta.data$class='Uncertain'
seurat2@meta.data[seurat2@meta.data$seurat_clusters %in% c(0,13,34),]$class='Astrocytes'
seurat2@meta.data[seurat2@meta.data$seurat_clusters %in% c(2,3,4,7,8,14,22,25,32),]$class='Oligo'
seurat2@meta.data[seurat2@meta.data$seurat_clusters %in% c(20),]$class='Endothelial'
seurat2@meta.data[seurat2@meta.data$seurat_clusters %in% c(1,27,36,31,33),]$class='OPC'
seurat2@meta.data[seurat2@meta.data$seurat_clusters %in% c(10),]$class='Microglia'
seurat2@meta.data[seurat2@meta.data$seurat_clusters %in% c(5,6,9,11,15,17,19,18,23,24,28,26,29,30,35,37),]$class='Neurons'
seurat2@meta.data[seurat2@meta.data$seurat_clusters %in% c(16),]$class='Ependymal'
# seurat2@meta.data[seurat2@meta.data$seurat_clusters %in% c(11),]$class='5-HT'

In [ ]:
colnames(seurat2@meta.data)

In [ ]:
# # 1. Create a new column, default "Unknown"
# seurat2@meta.data$subclass3 <- "Unknown"

# ## 2. Mapping table (cluster -> subclass)
# cls.map <- list(
#   Astrocytes  = c(1,18,20,36),
#   Oligo       = c(0,3,4,7,10,16),
#   Endothelial = 23,
#   OPC         = c(5,22,31,34),
#   Microglia   = c(9,17),
#   Neurons     = c(2,6,8,13,14,19,21,24,25,27,28,30,32,35),
#   `DA`        = 33,
#   `5-HT`      = c(11)
# )

# ## 3. Write in one pass
# for (nm in names(cls.map)) {
#   idx <- seurat2@meta.data$seurat_clusters %in% cls.map[[nm]]
#   if (sum(idx) > 0) seurat2@meta.data$subclass3[idx] <- nm
# }

# ## 4. Check
# table(seurat2@meta.data$subclass3, useNA = "ifany")

In [ ]:
# View major classes
table(seurat2@meta.data$class)

# View subclasses
# table(seurat2@meta.data$subclass3)

# View together
head(seurat2@meta.data[, c("seurat_clusters", "class")])

In [ ]:
df_test=seurat2@reductions$umap@cell.embeddings
df_test=as.data.frame(df_test)
df_test$X=rownames(df_test)

df_meta=seurat2@meta.data
df_meta=as.data.frame(df_meta)
df_meta$X=rownames(df_meta)
df_meta=df_meta[,c('nFeature_RNA','X','seurat_clusters')]

df_test=left_join(df_test,df_meta,by='X')


Idents(seurat2) = "seurat_clusters"
options(repr.plot.width=9, repr.plot.height=7)
DimPlot(seurat2, reduction="umap", label=T,raster=FALSE)+coord_fixed()

In [ ]:
classMarkerTop = c('SLC17A6','GAD1','GAD2','SNAP25','GFAP','PECAM1','FLT1','VCAN','MOBP','C1QC','FOXJ1','SLC6A4','TH')

options(repr.plot.width=20, repr.plot.height=15)
FeaturePlot(classFiltSeurat, features = classMarkerTop,raster=FALSE)
Idents(classFiltSeurat)='class'
DotPlot(classFiltSeurat, features = classMarkerTop)

In [ ]:
unique(seurat2@meta.data[seurat2@meta.data$class %in% 'Uncertain',]$seurat_clusters
)


In [ ]:
classFiltSeurat = RunHarmony(
classFiltSeurat, group.by.vars="donor", plot_convergence=TRUE,
assay.use="SCT", reduction.use = "pca", dims.use=1:30
)

In [ ]:
Idents(seurat2) = "class"
options(repr.plot.width=9, repr.plot.height=7)
DimPlot(seurat2, reduction="umap", label=T,raster=FALSE)+coord_fixed()

In [ ]:
classFiltSeurat=seurat1
classFiltSeurat

In [ ]:
# Count cells per class
cell_counts <- table(seurat2@meta.data$class)

# Print result
print(cell_counts)

In [ ]:
# Sample 80% per class, maximum 3000 cells per class
sampleRatio = 0.8
maxCell = 3000
sampleDf = seurat2@meta.data[, c("class"), drop=F]
sampleList = split(sampleDf, sampleDf$class)
sampleList = lapply(sampleList, function(x) {
    sampleSize = as.integer(nrow(x)*0.8)
    if (sampleSize > maxCell) {
        sampleSize = maxCell
    }
    sampleIdx = 1:nrow(x)
    sampleIdx = sample(1:nrow(x), size=sampleSize, replace=F)
    return(x[sampleIdx,,drop=F])
})
sampleDf = Reduce(rbind, sampleList)
str(sampleDf)
table(sampleDf$class)

trainSeurat = seurat2[,rownames(sampleDf)]
table(trainSeurat$class)

In [ ]:
trainSeurat$class %>% unique()

In [ ]:
Idents(trainSeurat) = "class"
classMarker = FindAllMarkers(trainSeurat)

In [ ]:
#classMarker = read_tsv(paste0(dataDir, "/allSnRNAseq.soupx.mergeSeuratFiltlibFilt800mt5Ratio1d2.downsample5000ClassMarker.20220822.tsv"))
classMarker = subset(classMarker, p_val_adj < 0.01 & avg_log2FC > 1)
table(classMarker$cluster)

classMarkerTop = classMarker %>% group_by(cluster) %>% top_n(n = 20, wt = avg_log2FC)
options(repr.plot.width=length(unique(classMarkerTop$gene))/3, repr.plot.height=5)
Idents(seurat) = "class"

In [ ]:
seurat3 = seurat
# View current group information
# levels(Idents(seurat3))

# Reorder groups, e.g., by a specific order
new_levels <- c("Endothelial",'OPC','Microglia','Oligo','Astrocytes','Neurons','Uncertain')
Idents(seurat3) <- factor(Idents(seurat3), levels = new_levels)

# Verify new group order
levels(Idents(seurat3))

In [ ]:
trainDf = as.data.frame(t(as.matrix(trainSeurat[["SCT"]]@data[unique(classMarker$gene), ])))
trainDf$class = trainSeurat$class
trainDf = subset(trainDf, class != "Uncertain")
trainDf$class = factor(trainDf$class)
# We need to modify the name of the columns
colnames(trainDf) = paste0("col_", colnames(trainDf))
colnames(trainDf) = gsub("-", "_", colnames(trainDf))

rf = randomForest(col_class~., data=trainDf, ntree=1000)

In [ ]:
# Sample 80% per class, maximum 3000 cells per class
sampleRatio = 0.8
maxCell = 3000
sampleDf = seurat2@meta.data[, c("class"), drop=F]
sampleList = split(sampleDf, sampleDf$class)
sampleList = lapply(sampleList, function(x) {
    sampleSize = as.integer(nrow(x)*0.8)
    if (sampleSize > maxCell) {
        sampleSize = maxCell
    }
    sampleIdx = 1:nrow(x)
    sampleIdx = sample(1:nrow(x), size=sampleSize, replace=F)
    return(x[sampleIdx,,drop=F])
})
sampleDf = Reduce(rbind, sampleList)
str(sampleDf)
table(sampleDf$class)

trainSeurat = seurat2[,rownames(sampleDf)]
table(trainSeurat$class)

In [ ]:
dim(rf$confusion)
colnames(rf$confusion)
rownames(rf$confusion)

In [ ]:
# Load required packages
library(ggplot2)
library(pheatmap)

# Assume rf is your random forest model object
# rf <- readRDS(paste0(dataDir, "/allSnRNAseq.soupx.mergeSeuratFiltlibFilt800mt5Ratio1d2.class.randomForest.20220822.rds"))

# View confusion matrix structure
str(rf$confusion)

# Modify row and column names of the confusion matrix
# Assume row and column names are class labels
# rownames(rf$confusion) <- gsub("Endotheial", "Endothelial", rownames(rf$confusion))
# colnames(rf$confusion) <- gsub("Endotheial", "Endothelial", colnames(rf$confusion))

# View modified confusion matrix
str(rf$confusion)

# Set plotting parameters
options(repr.plot.width=10, repr.plot.height=5)
plot(rf)

# Compute confusion matrix proportions
confusionRatio <- t(apply(rf$confusion, 1, function(x) {x / sum(x)}))

# Set heatmap parameters
options(repr.plot.width=10, repr.plot.height=10)
pheatmap(confusionRatio, 
         breaks=seq(0, 1, 0.01), 
         cluster_rows=FALSE, 
         cluster_cols=FALSE, 
         border_color=NA, 
         cellwidth=30, 
         cellheight=30, 
         fontsize=15)

In [ ]:
#rf = readRDS(paste0(dataDir, "/allSnRNAseq.soupx.mergeSeuratFiltlibFilt800mt5Ratio1d2.class.randomForest.20220822.rds"))
str(rf$confusion)
options(repr.plot.width=10, repr.plot.height=5)
plot(rf)

confusionRatio = t(apply(rf$confusion, 1, function(x) {x/sum(x)}))
options(repr.plot.width=10, repr.plot.height=10)
pheatmap(confusionRatio, breaks=seq(0,1,0.01), cluster_rows=F, cluster_cols=F, border_color=NA, cellwidth=30, cellheight=30, fontsize=15)

In [ ]:
# Remove the class.error column
confusionMatrix <- rf$confusion[, -which(colnames(rf$confusion) == "class.error")]

# View matrix after removal
str(confusionMatrix)
# Compute confusion matrix proportions
confusionRatio = t(apply(confusionMatrix, 1, function(x) {x/sum(x)}))

# Draw heatmap
pheatmap(confusionRatio, breaks=seq(0,1,0.01), cluster_rows=F, cluster_cols=F, border_color=NA, cellwidth=30, cellheight=30, fontsize=15)

In [ ]:
# Load required libraries
library(ggplot2)
library(pheatmap)

# Assume rf is the loaded random forest classifier
# Uncomment below if you need to load data
# dataDir <- "your data directory path"
# rf <- readRDS(paste0(dataDir, "/allSnRNAseq.soupx.mergeSeuratFiltlibFilt800mt5Ratio1d2.class.randomForest.20220822.rds"))

# View confusion matrix structure
str(rf$confusion)

# Plot random forest confusion matrix
options(repr.plot.width = 10, repr.plot.height = 5)
plot(rf)

# Remove the 'class.error' column from the confusion matrix
confusion_matrix <- rf$confusion
if ("class.error" %in% colnames(confusion_matrix)) {
  confusion_matrix <- confusion_matrix[, !colnames(confusion_matrix) %in% c("class.error")]
}

# Compute confusion matrix ratios
confusionRatio <- t(apply(confusion_matrix, 1, function(x) { x / sum(x) }))

# Plot confusion matrix heatmap with values displayed
options(repr.plot.width = 10, repr.plot.height = 10)
pheatmap(
  confusionRatio,
  breaks = seq(0, 1, 0.01), 
  cluster_rows = FALSE, 
  cluster_cols = FALSE, 
  border_color = NA, 
  cellwidth = 30, 
  cellheight = 30, 
  fontsize = 15, 
  color = colorRampPalette(c("#e65200", "#fffde7"))(100), 
  show_rownames = TRUE, 
  show_colnames = TRUE, 
  main = "Confusion Matrix Heatmap", 
  legend = TRUE, 
  display_numbers = TRUE, 
  number_format = "%.2f" 
)

In [ ]:
# Apply the trainer to filter cells
seuratCountDf = as.data.frame(t(as.matrix(seurat[["SCT"]]@data[unique(classMarker$gene), ])))
colnames(seuratCountDf) = paste0("col_", colnames(seuratCountDf))
colnames(seuratCountDf) = gsub("-", "_", colnames(seuratCountDf))
seuratClassPred = predict(rf, seuratCountDf, type="prob")
str(seuratClassPred)

seuratClassPredTb = tibble(
    cellId = rownames(seuratClassPred),
    maxPredClassScore = apply(seuratClassPred, 1, max),
    maxPredClass = apply(seuratClassPred, 1, function(x) {names(x)[which.max(x)]})
)
head(seuratClassPredTb)

summary(seuratClassPredTb$maxPredClassScore)
options(repr.plot.width=6, repr.plot.height=4)
plot(density(seuratClassPredTb$maxPredClassScore, adjust=0.1))

identical(seuratClassPredTb$cellId, rownames(seurat@meta.data))
tmpDf = as.data.frame(seuratClassPredTb[c("maxPredClassScore", "maxPredClass")])
rownames(tmpDf) = seuratClassPredTb$cellId

seurat = AddMetaData(seurat, tmpDf)

options(repr.plot.width=9, repr.plot.height=7)
FeaturePlot(seurat, reduction="umap", features="maxPredClassScore",raster=FALSE)+coord_fixed()


In [ ]:
# 1. Check whether removed cells cluster in specific clusters
seurat$passFilt <- seurat$maxPredClassScore >= 0.9
table(seurat$seurat_clusters, seurat$passFilt)

# 2. View quality metrics of removed cells
VlnPlot(seurat, features = c("nFeature_RNA", "nCount_RNA", "percent.mt"),
        group.by = "passFilt", pt.size = 0)

In [ ]:
# 1. Check whether removed cells cluster in specific clusters
seurat$passFilt <- seurat$maxPredClassScore >= 0.8
table(seurat$seurat_clusters, seurat$passFilt)

# 2. View quality metrics of removed cells
VlnPlot(seurat, features = c("nFeature_RNA", "nCount_RNA", "percent.mt"),
        group.by = "passFilt", pt.size = 0)

In [ ]:
Idents(seurat) = "seurat_clusters"
cellNum0 = ncol(seurat)
classFiltSeurat = subset(seurat, maxPredClassScore >= 0.8)
cellNum1 = ncol(classFiltSeurat)
message("Filt cell ratio: ", cellNum1/cellNum0)

In [ ]:
seurat
classFiltSeurat

In [ ]:
table(classFiltSeurat$maxPredClass)

In [ ]:
saveRDS(classFiltSeurat,'./data/20260821subclass3macaque_snrna_batch1_soupX-forest.rds')

In [ ]:
classFiltSeurat=readRDS('./data/20260821subclass3macaque_snrna_batch1_soupX-forest.rds')

In [ ]:
# seurat

# # 2. View meta.data columns (look for class, subclass, maxPredClassScore, maxPredClass, passFilt)
# colnames(seurat@meta.data)

# # 3. Current total cell count
# ncol(seurat)

# # 4. Major class annotation distribution (if class column still exists)
# table(seurat@meta.data$class, useNA = "ifany")

# # 5. How many Uncertain remain, and which clusters
# table(seurat@meta.data[seurat@meta.data$class == "Uncertain", ]$seurat_clusters)
# # Or view Uncertain subclass distribution directly:
# table(seurat@meta.data[seurat@meta.data$class == "Uncertain", ]$subclass, useNA = "ifany")

# # 6. Random forest prediction overview (if stored)
# summary(seurat@meta.data$maxPredClassScore)
# table(seurat@meta.data$maxPredClass, useNA = "ifany")

# # 7. If already filtered at 0.8, view post-filter status
# if ("passFilt" %in% colnames(seurat@meta.data)) {
#   table(seurat@meta.data$passFilt)
#   table(seurat@meta.data[seurat@meta.data$passFilt == FALSE, ]$seurat_clusters)
# }

In [ ]:
# ncol(seurat)                                   # Expected: 123,245 (batch1)
# colnames(seurat@meta.data)                     # Check for class / maxPredClassScore columns
# table(seurat@meta.data$class, useNA = "ifany")
# table(seurat@meta.data[seurat@meta.data$class == "Uncertain", ]$seurat_clusters)

In [ ]:
# Step 4: annotate subclasses

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)   # Added here

In [ ]:
library(Seurat)
library(dplyr)

# ========== 1. Extract Neurons directly from maxPredClass ==========
nd_seurat <- subset(classFiltSeurat, subset = maxPredClass == "Neurons")
cat("Number of Neuron cells:", ncol(nd_seurat), "\n")

# ========== 2. Recompute variable features + PCA (Neurons-specific) ==========
nd_seurat <- FindVariableFeatures(nd_seurat, selection.method = "vst", nfeatures = 3000)
nd_seurat <- RunPCA(nd_seurat, features = VariableFeatures(nd_seurat), npcs = 50, verbose = FALSE)

# ========== 3. Clustering + UMAP ==========
nd_seurat <- FindNeighbors(nd_seurat, reduction = "pca", dims = 1:30)
nd_seurat <- FindClusters(nd_seurat, resolution = 0.2)   # Adjust as needed
nd_seurat <- RunUMAP(nd_seurat, reduction = "pca", dims = 1:30)

cat("New cluster distribution:\n")
print(table(nd_seurat$seurat_clusters))

# ========== 4. Rename as Neurons_0, Neurons_1... ==========
clusters <- as.character(nd_seurat$seurat_clusters)
unique_clusters <- sort(unique(as.numeric(clusters)))

neuron_counter <- 0
cluster_map <- character(length(unique_clusters))
names(cluster_map) <- as.character(unique_clusters)

for(cl in as.character(unique_clusters)) {
  cluster_map[cl] <- paste0("Neurons_", neuron_counter)
  neuron_counter <- neuron_counter + 1
}

nd_seurat$Subclass_new <- unname(cluster_map[clusters])
cat("\nAfter renaming:\n")
print(table(nd_seurat$Subclass_new))

# ========== 5. Visualization ==========
options(repr.plot.width = 14, repr.plot.height = 10)
DimPlot(nd_seurat, reduction = "umap", group.by = "Subclass_new",
        label = TRUE, repel = TRUE, raster = FALSE) + coord_fixed()

In [ ]:
# 1. Create cluster column in classFiltSeurat, default to major class name
classFiltSeurat$cluster <- as.character(classFiltSeurat$maxPredClass)

# 2. Write Neuron subclass labels back
nd_cells <- colnames(nd_seurat)
classFiltSeurat$cluster[nd_cells] <- as.character(nd_seurat$Subclass_new)

# 3. Check
table(classFiltSeurat$cluster)

In [ ]:
output_path <- "./data/"

In [ ]:
print(output_path)

In [ ]:
# ========== Place before all plotting code ==========
dir.create(output_path, showWarnings = FALSE, recursive = TRUE)


In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)

# ========== Path settings ==========
output_path <- "./data/"
dir.create(output_path, showWarnings = FALSE, recursive = TRUE)

# ============================================
# Extract Neuron subclasses only (using previously created nd_seurat)
# ============================================

# Make sure nd_seurat exists and contains Subclass_new
# If unsure, re-extract from classFiltSeurat:
# nd_seurat <- subset(classFiltSeurat, subset = grepl("^Neurons_", cluster))

cat("Number of Neuron cells:", ncol(nd_seurat), "\n")
cat("Number of Neuron subclasses:", length(unique(nd_seurat$Subclass_new)), "\n")

# ============================================
# 1. UMAP overview (Neuron subclasses)
# ============================================
cat("\n--- 1. Neuron UMAP ---\n")

p_umap <- DimPlot(nd_seurat, reduction = "umap", group.by = "Subclass_new",
                  label = TRUE, repel = TRUE, raster = FALSE, pt.size = 0.3) +
  coord_fixed() +
  ggtitle("Neurons Subclusters") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 14))

print(p_umap)
ggsave(file.path(output_path, "Neurons_only_UMAP.png"), 
       plot = p_umap, width = 14, height = 10, dpi = 300)

# ============================================
# 2. Classic marker genes
# ============================================
cat("\n--- 2. Classic Markers (Neurons only) ---\n")

marker_genes <- c("SLC17A6","SLC17A7" ,"SLC6A4", "TPH2", "GAD1", "GAD2", "TH", "SLC17A8", "SNAP25",'GFAP','PECAM1','FLT1','VCAN','MOBP','C1QC','FOXJ1')
avail_genes <- intersect(marker_genes, rownames(nd_seurat))

if (length(avail_genes) > 0) {
  
  # FeaturePlot (on the Neuron UMAP)
  p_feature <- FeaturePlot(nd_seurat, features = avail_genes, ncol = 3,
                           pt.size = 0.3, cols = c("lightgrey", "darkblue"),
                           order = TRUE, raster = FALSE) &
    theme(plot.title = element_text(face = "bold.italic", size = 12))
  print(p_feature)
  ggsave(file.path(output_path, "Neurons_only_FeaturePlot.png"), 
         plot = p_feature, width = 15, height = 12, dpi = 300)
  
  # DotPlot (grouped by Neuron subclasses)
  # 56 subclasses; y-axis is long, so increase height
  p_dot_classic <- DotPlot(nd_seurat, features = avail_genes, 
                           group.by = "Subclass_new",
                           dot.scale = 8, cols = c("lightgrey", "darkblue")) +
    RotatedAxis() +
    ggtitle("Neurons Classic Markers") +
    theme(axis.text.x = element_text(angle = 45, hjust = 1, size = 11, face = "italic"),
          axis.text.y = element_text(size = 6),   # 56 subclasses; shrink font
          plot.title = element_text(hjust = 0.5, face = "bold", size = 13))
  print(p_dot_classic)
  ggsave(file.path(output_path, "Neurons_only_ClassicMarkers_DotPlot.png"), 
         plot = p_dot_classic, width = 10, height = 20, dpi = 300)  # Increased height
  
  # VlnPlot (grouped by Neuron subclasses)
  p_vln <- VlnPlot(nd_seurat, features = avail_genes, 
                   group.by = "Subclass_new",
                   pt.size = 0, ncol = 3) &
    theme(axis.text.x = element_text(angle = 45, hjust = 1, size = 7),
          axis.text.y = element_text(size = 6),
          plot.title = element_text(face = "bold.italic", size = 12))
  print(p_vln)
  ggsave(file.path(output_path, "Neurons_only_ClassicMarkers_VlnPlot.png"), 
         plot = p_vln, width = 16, height = 24, dpi = 300)  # Increased height
  
  cat("OK: Neuron classic marker plots saved\n")
} else {
  cat("WARNING: marker genes not found in expression matrix\n")
}

In [ ]:
# ============================================
# 3. FindAllMarkers (comparison among Neuron subclasses)
# ============================================
cat("\n--- 3. FindAllMarkers (Neurons only) ---\n")

DefaultAssay(nd_seurat) <- "RNA"
Idents(nd_seurat) <- "Subclass_new"

markers <- FindAllMarkers(
  nd_seurat,
  assay = "RNA",
  only.pos = TRUE,
  min.pct = 0.25,
  logfc.threshold = 0.25,
  verbose = TRUE
)
cat("Total markers found:", nrow(markers), "\n")

# Save
write.csv(markers, file.path(output_path, "Neurons_only_FindAllMarkers.csv"), row.names = FALSE)

# ============================================
# 4. Top20 Markers DotPlot
# ============================================
cat("\n--- 4. Top20 Markers DotPlot ---\n")

if (nrow(markers) > 0) {
  top20 <- markers %>%
    group_by(cluster) %>%
    slice_max(order_by = avg_log2FC, n = 20) %>%
    ungroup()
  
  top20_genes <- unique(top20$gene)
  top20_genes <- top20_genes[top20_genes %in% rownames(nd_seurat)]
  
  if (length(top20_genes) > 0) {
    gene_order <- top20 %>% 
      filter(gene %in% top20_genes) %>% 
      arrange(cluster) %>% 
      pull(gene) %>% 
      unique()
    
    p_dot_top20 <- DotPlot(
      nd_seurat,
      features = gene_order,
      group.by = "Subclass_new",
      dot.scale = 4,
      cols = c("lightgrey", "red")
    ) +
      RotatedAxis() +
      ggtitle("Neurons Top20 Markers per Subcluster") +
      theme(
        axis.text.x = element_text(angle = 45, hjust = 1, size = 7),
        axis.text.y = element_text(size = 6),
        plot.title = element_text(hjust = 0.5, face = "bold", size = 12)
      )
    
    print(p_dot_top20)
    ggsave(
      file.path(output_path, "Neurons_only_Top20Markers_DotPlot.png"),
      plot = p_dot_top20,
      width = 50,
      height = 24,
      dpi = 300,
      limitsize = FALSE
    )
    cat("OK: Top20 marker dot plot saved\n")
  }
}

In [ ]:
# ==========================================================================
# Neuron subclass marker violin plots —— complete script
# For issues: VlnPlot 'S4SXP' error + no graphics device in headless environments
# Solution: FetchData extraction + manual ggplot2 + ggsave with PDF priority
# Usage: source or run with Rscript after restarting the R session
# ==========================================================================

suppressPackageStartupMessages({
  library(Seurat)
  library(ggplot2)
  library(dplyr)
  library(tidyr)
})

# ============================ 0. Parameters (modify as needed) ============================
group_col   <- "Subclass_new"
output_path <- "./data/"
markers_csv <- file.path(output_path, "Neurons_only_FindAllMarkers.csv")
# nd_seurat_rds <- "/path/to/nd_seurat.rds"   # If object not in memory, fill in path and uncomment next line
# nd_seurat <- readRDS(nd_seurat_rds)

dir.create(output_path, showWarnings = FALSE, recursive = TRUE)

# ============================ 1. Object check ============================
if (!exists("nd_seurat")) stop("nd_seurat not in memory; please readRDS first")
stopifnot(group_col %in% colnames(nd_seurat@meta.data))
Idents(nd_seurat) <- group_col

cat("== Environment diagnostics ==\n")
cat("Interactive:", interactive(), " X11:", capabilities("X11"), " cairo:", capabilities("cairo"), "\n")
cat("Cells:", ncol(nd_seurat), " Genes:", nrow(nd_seurat), "\n")
cat("Group levels:", paste(sort(unique(nd_seurat@meta.data[[group_col]])), collapse = ", "), "\n")

if (!exists("markers")) {
  if (file.exists(markers_csv)) {
    markers <- read.csv(markers_csv, stringsAsFactors = FALSE)
    cat("Markers read from CSV:", nrow(markers), "rows\n")
  } else {
    stop("markers not in memory and file not found: ", markers_csv)
  }
}

# ============================ 2. Marker panel definition ============================

# ---- 2.1 Neurotransmitter identity panel ----
nt_markers <- c(
  "SLC17A6", "SLC17A7", "SLC17A8",   # Glut (SLC17A8 is also a 5-HT co-release marker)
  "GAD1",    "GAD2",    "SLC32A1",   # GABA
  "CHAT",    "SLC18A3", "SLC5A7",    # Cholinergic
  "TH",      "SLC6A3",  "NR4A2",     # Dopaminergic
  "TPH2",    "SLC6A4",  "DDC",       # 5-HT
  "DBH",     "SLC6A2"                # Noradrenergic (LC direction)
)

# ---- 2.2 Adjacent-region identity marker panel ----
region_mk_list <- list(
  `5HT_raphe`   = c("TPH2","SLC6A4","FEV","SLC17A8","LMX1B","GATA2"),
  `PAG`         = c("PAX7","PAX8","ESR1","NTN1","DPP10","GUCY1A3"),
  `PBN`         = c("EGR2","PRLR","CALCA","NMB","DRD3"),
  `LC_NE`       = c("DBH","SLC6A2","PHOX2B","GATA3"),
  `VTA_SNc_DA`  = c("TH","NR4A2","PITX3","SLC6A3","SOX6","KCNJ6"),
  `ACh_IPN_MHb` = c("CHAT","SLC18A3","SLC5A7","GPR151","TACR3"),
  `Striatum`    = c("BCL11B","FOXP2","PTPN5","TAC1","ADORA2A","DRD1","DRD2"),
  `Ctx_Hip_GABA`= c("PVALB","SST","VIP","LAMP5","NPY")
)

# ============================ 3. Core functions ============================

# ---- 3.1 Extract data: gene expression or meta.data columns (incl. score columns) to long format ----
get_expr_df <- function(features, obj = nd_seurat, gc = group_col) {
  dat <- tryCatch(
    FetchData(obj, vars = c(features, gc)),
    error = function(e) NULL
  )
  if (is.null(dat)) {
    # Fallback if FetchData fails: extract directly from expression matrix (genes only, no meta columns)
    mat <- tryCatch(GetAssayData(obj, assay = "RNA", layer = "data"),
                    error = function(e) GetAssayData(obj, assay = "RNA", slot = "data"))
    genes <- intersect(features, rownames(mat))
    dat <- as.data.frame(t(as.matrix(mat[genes, , drop = FALSE])))
    dat[[gc]] <- obj@meta.data[[gc]]
  }
  feats_ok <- intersect(features, colnames(dat))
  dat %>%
    select(all_of(c(feats_ok, gc))) %>%
    pivot_longer(cols = all_of(feats_ok), names_to = "gene", values_to = "expr")
}

# ---- 3.2 Draw violin plots and save (PDF always, PNG attempted) ----
plot_vln_manual <- function(features, file_base, ncol = 4, same.y = TRUE,
                            title = NULL, width = NULL, height = NULL) {
  df <- get_expr_df(features)
  genes <- intersect(features, unique(df$gene))
  if (length(genes) == 0) {
    cat("!! No usable genes, skipping:", basename(file_base), "\n")
    return(invisible(NULL))
  }
  df <- df %>% mutate(gene = factor(gene, levels = genes))
  n_row <- ceiling(length(genes) / ncol)
  if (is.null(width))  width  <- 3.0 * ncol
  if (is.null(height)) height <- 2.8 * n_row

  p <- ggplot(df, aes(x = .data[[group_col]], y = expr, fill = .data[[group_col]])) +
    geom_violin(scale = "width", trim = TRUE) +
    facet_wrap(~gene, ncol = ncol, scales = ifelse(same.y, "fixed", "free_y")) +
    theme_bw(base_size = 8) +
    theme(axis.text.x = element_text(angle = 45, hjust = 1),
          legend.position = "none",
          strip.background = element_rect(fill = "grey90")) +
    labs(x = NULL, y = "Expression", title = title)

  ggsave(paste0(file_base, ".pdf"), p, width = width, height = height, limitsize = FALSE)
  ok_png <- tryCatch({
    ggsave(paste0(file_base, ".png"), p, width = width, height = height,
           dpi = 300, limitsize = FALSE)
    TRUE
  }, error = function(e) FALSE)
  cat(">> Done:", basename(file_base), "(pdf saved", ifelse(ok_png, ", png saved)", ")"), "\n")
  invisible(p)
}

# ---- 3.3 Region scoring (manual fallback if AddModuleScore fails) ----
add_scores_safe <- function(obj, sig_list) {
  obj <- tryCatch({
    obj <- AddModuleScore(obj, features = sig_list, name = "RS", seed = 1)
    for (i in seq_along(sig_list)) {
      obj@meta.data[[names(sig_list)[i]]] <- obj@meta.data[[paste0("RS", i)]]
    }
    cat("AddModuleScore completed successfully\n")
    obj
  }, error = function(e) {
    cat("!! AddModuleScore failed, falling back to manual scoring:", conditionMessage(e), "\n")
    for (nm in names(sig_list)) {
      genes <- intersect(sig_list[[nm]], rownames(obj))
      if (length(genes) == 0) next
      expr <- FetchData(obj, vars = genes)
      obj@meta.data[[nm]] <- rowMeans(expr)
    }
    obj
  })
  obj
}

# ============================ 4. Generate plots ============================

# ---- 4.1 Neurotransmitter identity overview ----
cat("\n--- 4.1 Neurotransmitter identity violin plots ---\n")
plot_vln_manual(nt_markers,
                file.path(output_path, "Neurons_only_NeuroTransmitter_VlnPlot"),
                ncol = 4, same.y = TRUE,
                title = "Neurotransmitter identity")

# ---- 4.2 Adjacent-region markers ----
cat("\n--- 4.2 Adjacent-region marker violin plots ---\n")
plot_vln_manual(unique(unlist(region_mk_list)),
                file.path(output_path, "Neurons_only_RegionMarkers_VlnPlot"),
                ncol = 5, same.y = FALSE,
                title = "Adjacent-region markers")

# ---- 4.3 Region scoring ----
cat("\n--- 4.3 Region scoring ---\n")
nd_seurat <- add_scores_safe(nd_seurat, region_mk_list)
plot_vln_manual(names(region_mk_list),
                file.path(output_path, "Neurons_only_RegionScore_VlnPlot"),
                ncol = 4, same.y = FALSE,
                title = "Region module scores")

# ---- 4.4 Top20 markers per subclass ----
cat("\n--- 4.4 Top20 markers per subclass ---\n")
top20 <- markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 20, with_ties = FALSE) %>%
  ungroup()

for (cl in sort(unique(top20$cluster))) {
  genes_cl <- top20 %>%
    filter(cluster == cl) %>%
    arrange(desc(avg_log2FC)) %>%
    pull(gene)
  ncol_use <- min(5, max(length(intersect(genes_cl, rownames(nd_seurat))), 1))
  nrow_use <- ceiling(length(intersect(genes_cl, rownames(nd_seurat))) / ncol_use)
  plot_vln_manual(genes_cl,
                  file.path(output_path, paste0("Neurons_only_Top20_VlnPlot_", cl)),
                  ncol = ncol_use, same.y = FALSE,
                  width = 3.2 * ncol_use, height = 3.2 * nrow_use,
                  title = paste0("Top20 markers of ", cl))
}

# ============================ 5. Completion check ============================
cat("\n== All done. Output files: ==\n")
print(list.files(output_path, pattern = "VlnPlot", full.names = TRUE))

In [ ]:
library(Seurat)
library(dplyr)
library(pheatmap)

group_col <- "Subclass_new"
clusters_check <- c("Neurons_5","Neurons_9","Neurons_10","Neurons_11","Neurons_12",
                    "Neurons_13","Neurons_14","Neurons_15","Neurons_16","Neurons_17",
                    "Neurons_18","Neurons_20","Neurons_21","Neurons_22","Neurons_8")

sigs <- list(
  Glut   = c("SLC17A6","SLC17A7"),
  GABA   = c("GAD1","GAD2","SLC32A1"),
  DA     = c("TH","SLC6A3","NR4A2","PITX3","LMX1A"),
  X5HT   = c("TPH2","SLC6A4","FEV","LMX1B","GATA2","SLC17A8","DDC"),
  NE     = c("DBH","SLC6A2","PHOX2B","GATA3"),
  ACh    = c("CHAT","SLC18A3","SLC5A7"),
  PAG        = c("PAX7","PAX8","ESR1","DPP10","NTN1"),
  PBN        = c("EGR2","PRLR","CALCA","NMB","PDYN"),
  RTN        = c("SHOX2","PHOX2B","MEIS1"),
  PonsOlive  = c("EBF2","EBF1","OTOF","NFIB","TAC1","CALB2"),
  Striatum   = c("PTPN5","PDE1B","TAC1","PENK","FOXP2","BCL11B","ADORA2A"),
  Oligo   = c("PLP1","MBP","OPALIN","MOBP","MAG"),
  Astro   = c("AQP4","SLC1A3","GJA1","ALDH1A1","SPARCL1"),
  Endo    = c("CLDN5","VWF","FLT1","PECAM1"),
  Micro   = c("AIF1","CSF1R","CX3CR1","C1QA","FCGR3A"),
  ChoriEpend = c("TTR","KRT8","KRT18","FOXJ1"),
  Pericyte= c("RGS5","ABCC9","KCNJ8","PDGFRB")
)

# ---- Average expression (v4/v5 compatible) ----
all_genes <- unique(unlist(sigs))
missing_genes <- setdiff(all_genes, rownames(nd_seurat))
if (length(missing_genes) > 0) {
  cat("The following genes are not in the RNA assay and were removed from signatures:",
      paste(missing_genes, collapse = ", "), "\n")
}

avg <- tryCatch(
  AverageExpression(nd_seurat, assays = "RNA", layer = "data",
                    features = all_genes, group.by = group_col)$RNA,
  error = function(e)
    AverageExpression(nd_seurat, assays = "RNA", slot = "data",
                      features = all_genes, group.by = group_col)$RNA
)

# ---- Key fix: restore "_" replaced by Seurat ----
colnames(avg) <- gsub("-", "_", colnames(avg))

matched <- intersect(clusters_check, colnames(avg))
cat("Matched clusters:", length(matched), "/", length(clusters_check), "\n")
not_matched <- setdiff(clusters_check, colnames(avg))
if (length(not_matched) > 0) {
  cat("Not matched (IDs may not exist or be misspelled):", paste(not_matched, collapse = ", "), "\n")
}
stopifnot("At least one cluster matched" = length(matched) > 0)

avg <- avg[, matched, drop = FALSE]

# ---- z-score ----
m <- as.matrix(avg)
z <- t(scale(t(m)))
z[!is.finite(z)] <- 0   # Handle NaN caused by identical expression across all clusters

# ---- Signature scores ----
sig_score <- sapply(sigs, function(g) {
  g <- intersect(g, rownames(z))
  if (length(g) == 0) return(setNames(rep(0, ncol(z)), colnames(z)))
  v <- colMeans(z[g, , drop = FALSE], na.rm = TRUE)
  v[is.na(v)] <- 0
  v
})
sig_score <- t(sig_score)
sig_score[is.na(sig_score)] <- 0

# ---- Automated annotation table ----
transmitter <- c("Glut","GABA","DA","X5HT","NE","ACh")
region      <- c("PAG","PBN","RTN","PonsOlive","Striatum")
glial       <- c("Oligo","Astro","Endo","Micro","ChoriEpend","Pericyte")

anno <- lapply(colnames(sig_score), function(cl) {
  tibble(
    cluster      = cl,
    transmitter  = transmitter[which.max(sig_score[transmitter, cl])],
    region_hit   = region[which.max(sig_score[region, cl])],
    region_score = round(max(sig_score[region, cl]), 2),
    glial_hit    = glial[which.max(sig_score[glial, cl])],
    glial_score  = round(max(sig_score[glial, cl]), 2),
    verdict = case_when(
      max(sig_score[glial, cl]) > 1.0   ~ paste0("Non-neuronal/doublet (", glial[which.max(sig_score[glial, cl])], ")"),
      max(sig_score[region, cl]) > 1.0   ~ paste0("Suspected ", region[which.max(sig_score[region, cl])], " contamination"),
      max(sig_score[transmitter, cl]) < 0 ~ "All transmitter markers negative, manual review needed",
      TRUE                                ~ "Likely local raphe population"
    )
  )
}) %>% bind_rows()

print(as.data.frame(anno))
write.csv(anno, file.path(output_path, "cluster_auto_annotation.csv"), row.names = FALSE)

# ---- Heatmap ----
pdf(file.path(output_path, "cluster_signature_heatmap.pdf"), width = 11, height = 8)
pheatmap(sig_score, fontsize_row = 8, fontsize_col = 8,
         main = "Signature scores by cluster (row z-score)",
         cluster_cols = FALSE)
dev.off()
cat("Done. Results saved.\n")

In [ ]:
# ============================================
# 3. FindAllMarkers (comparison among Neuron subclasses)
# ============================================
# cat("\n--- 3. FindAllMarkers (Neurons only) ---\n")

# DefaultAssay(nd_seurat) <- "RNA"
# Idents(nd_seurat) <- "Subclass_new"

# markers <- FindAllMarkers(
#   nd_seurat,
#   assay = "RNA",
#   only.pos = TRUE,
#   min.pct = 0.25,
#   logfc.threshold = 0.25,
#   verbose = TRUE
# )
# cat("Total markers found:", nrow(markers), "\n")

# Save
# write.csv(markers, file.path(output_path, "Neurons_only_FindAllMarkers.csv"), row.names = FALSE)

# ============================================
# 4. Top20 Markers DotPlot
# ============================================
# cat("\n--- 4. Top20 Markers DotPlot ---\n")

# if (nrow(markers) > 0) {
#   top20 <- markers %>%
#     group_by(cluster) %>%
#     slice_max(order_by = avg_log2FC, n = 20) %>%
#     ungroup()
#   
#   top20_genes <- unique(top20$gene)
#   top20_genes <- top20_genes[top20_genes %in% rownames(nd_seurat)]
#   
#   if (length(top20_genes) > 0) {
#     gene_order <- top20 %>% 
#       filter(gene %in% top20_genes) %>% 
#       arrange(cluster) %>% 
#       pull(gene) %>% 
#       unique()
#     
#     p_dot_top20 <- DotPlot(
#       nd_seurat,
#       features = gene_order,
#       group.by = "Subclass_new",
#       dot.scale = 4,
#       cols = c("lightgrey", "red")
#     ) +
#       RotatedAxis() +
#       ggtitle("Neurons Top20 Markers per Subcluster") +
#       theme(
#         axis.text.x = element_text(angle = 45, hjust = 1, size = 7),
#         axis.text.y = element_text(size = 6),
#         plot.title = element_text(hjust = 0.5, face = "bold", size = 12)
#       )
#     
#     print(p_dot_top20)
#     ggsave(
#       file.path(output_path, "Neurons_only_Top20Markers_DotPlot.png"),
#       plot = p_dot_top20,
#       width = 50,
#       height = 24,
#       dpi = 300,
#       limitsize = FALSE
#     )
#     cat("OK: Top20 marker dot plot saved\n")
#   }
# }

In [ ]:
# From markers: Raphe-adjacent (NE)  Neurons_22  NE  DBH, PHOX2A, SLC6A2, TH  Locus coeruleus or A1/A2 nuclei
# Raphe-adjacent (ACh)  Neurons_13, 18  ACh  CHAT, SLC5A7  PPN/LDT cholinergic nuclei
# Raphe-adjacent (PAG)  Neurons_6, 8  PAG  PAX3, PAX7, OTP  Periaqueductal gray

In [ ]:
library(Seurat)
library(harmony)

In [ ]:
library(Seurat)
library(dplyr)

# ========== 1. Define Neuron subclasses to remove ==========
remove_neurons <- c("Neurons_11", "Neurons_12", "Neurons_13", "Neurons_16", "Neurons_22","Neurons_20", "Neurons_9", "Neurons_5", "Neurons_14", "Neurons_10")

# ========== 2. Remove these cells from classFiltSeurat ==========
cells_to_remove <- colnames(classFiltSeurat)[classFiltSeurat$cluster %in% remove_neurons]
cat("Cells to be removed:", length(cells_to_remove), "\n")

# Create new Seurat object (removing specified cells)
classFiltSeurat_filt <- subset(classFiltSeurat, cells = setdiff(colnames(classFiltSeurat), cells_to_remove))

cat("Total cells before removal:", ncol(classFiltSeurat), "\n")
cat("Total cells after removal:", ncol(classFiltSeurat_filt), "\n")

# ========== 3. View cluster distribution after removal ==========
cat("\nCluster distribution after removal:\n")
print(table(classFiltSeurat_filt$cluster))

# ========== 4. Re-embed Neuron subset (optional, if re-integration is needed) ==========
# Extract Neuron cells
neuron_clusters <- grep("^Neurons_", unique(classFiltSeurat_filt$cluster), value = TRUE)
nd_cells_filt <- colnames(classFiltSeurat_filt)[classFiltSeurat_filt$cluster %in% neuron_clusters]

nd_seurat_filt <- subset(classFiltSeurat_filt, cells = nd_cells_filt)

cat("\nRemaining Neuron cells:", ncol(nd_seurat_filt), "\n")

# Recompute variable features + PCA
nd_seurat_filt <- FindVariableFeatures(nd_seurat_filt, selection.method = "vst", nfeatures = 3000)
nd_seurat_filt <- RunPCA(nd_seurat_filt, features = VariableFeatures(nd_seurat_filt), npcs = 50, verbose = FALSE)

# New: donor batch correction
nd_seurat_filt <- RunHarmony(nd_seurat_filt,
                             group.by.vars = "donor",
                             reduction.use = "pca",
                             dims.use = 1:30,
                             assay.use = "RNA",
                             verbose = FALSE)

nd_seurat_filt <- FindNeighbors(nd_seurat_filt, reduction = "harmony", dims = 1:30)
nd_seurat_filt <- FindClusters(nd_seurat_filt, resolution = 0.2)
nd_seurat_filt <- RunUMAP(nd_seurat_filt, reduction = "harmony", dims = 1:30)

# Rename as Neurons_0, Neurons_1... (keep numbering contiguous)
clusters <- as.character(nd_seurat_filt$seurat_clusters)
unique_clusters <- sort(unique(as.numeric(clusters)))

neuron_counter <- 0
cluster_map <- character(length(unique_clusters))
names(cluster_map) <- as.character(unique_clusters)

for(cl in as.character(unique_clusters)) {
  cluster_map[cl] <- paste0("Neurons_", neuron_counter)
  neuron_counter <- neuron_counter + 1}

nd_seurat_filt$Subclass_new <- unname(cluster_map[clusters])

cat("\nDistribution after re-clustering:\n")
print(table(nd_seurat_filt$Subclass_new))

# ========== 5. Update Neuron labels in classFiltSeurat_filt ==========
# Non-neuron cells keep original labels
classFiltSeurat_filt$cluster_final <- as.character(classFiltSeurat_filt$cluster)

# Update new labels for Neuron cells
nd_cells_new <- colnames(nd_seurat_filt)
classFiltSeurat_filt$cluster_final[nd_cells_new] <- as.character(nd_seurat_filt$Subclass_new)

cat("\nFinal cluster distribution:\n")
print(table(classFiltSeurat_filt$cluster_final))

# ========== 6. Visualization ==========
# Neuron subset UMAP
options(repr.plot.width = 12, repr.plot.height = 10)
DimPlot(nd_seurat_filt, reduction = "umap", group.by = "Subclass_new",
        label = TRUE, repel = TRUE, raster = FALSE) + coord_fixed()

# All-cell UMAP (colored by final cluster)
# Re-run UMAP on all cells, or use the existing one
# Code below re-runs UMAP based on the existing PCA:
classFiltSeurat_filt <- RunUMAP(classFiltSeurat_filt, reduction = "pca", dims = 1:30)

options(repr.plot.width = 14, repr.plot.height = 10)
DimPlot(classFiltSeurat_filt, reduction = "umap", group.by = "cluster_final",
        label = TRUE, repel = TRUE, raster = FALSE) + coord_fixed()



In [ ]:
# ========== 7. Save objects (optional) ==========
saveRDS(classFiltSeurat_filt, file = "classFiltSeurat_filt.rds")
saveRDS(nd_seurat_filt, file = "nd_seurat_filt.rds")

In [ ]:
library(Seurat)
library(ggplot2)
library(patchwork)

# ========== Gene list ==========
genes <- c("SLC17A6", "SLC17A7", "SLC6A4", "TPH2", "GAD1", "GAD2", 
           "TH", "SLC17A8", "SNAP25", "GFAP", "PECAM1", "FLT1", 
           "VCAN", "MOBP", "C1QC", "FOXJ1")

# Check which genes are present in the data
genes_avail <- genes[genes %in% rownames(nd_seurat_filt)]
genes_missing <- setdiff(genes, rownames(nd_seurat_filt))
if (length(genes_missing) > 0) {
  cat("The following genes are not in the data (check case or species):", paste(genes_missing, collapse = ", "), "\n")
}
cat("Available genes:", paste(genes_avail, collapse = ", "), "\n")

# ============================================================
# I. Plots for Neuron subset (nd_seurat_filt)
# ============================================================

# -------- 1.1 DotPlot (by Neuron subclasses) --------
p_dot_neurons <- DotPlot(nd_seurat_filt, features = genes_avail, group.by = "Subclass_new") +
  coord_flip() +
  scale_color_gradient2(low = "blue", mid = "white", high = "red", midpoint = 0) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  ggtitle("Neuron Subclusters - Marker DotPlot")

print(p_dot_neurons)

In [ ]:
# -------- 1.2 FeaturePlot (UMAP) --------
# Split into two groups to avoid overcrowding
p_feat_neurons_1 <- FeaturePlot(nd_seurat_filt, features = genes_avail[1:8], 
                                  cols = c("lightgrey", "darkred"), 
                                  raster = FALSE, ncol = 4)
p_feat_neurons_2 <- FeaturePlot(nd_seurat_filt, features = genes_avail[9:length(genes_avail)], 
                                  cols = c("lightgrey", "darkred"), 
                                  raster = FALSE, ncol = 4)

print(p_feat_neurons_1)
print(p_feat_neurons_2)

# -------- 1.3 5-HT core markers shown separately --------
ht_core <- c("TPH2", "SLC6A4")
if (all(ht_core %in% genes_avail)) {
  p_ht <- FeaturePlot(nd_seurat_filt, features = ht_core, 
                      cols = c("lightgrey", "darkred"), 
                      raster = FALSE, ncol = 2) &
    theme(plot.title = element_text(face = "bold"))
  print(p_ht)
}


In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)

# ============================================================
# Step 1: Label 5-HT vs non-5-HT neurons in nd_seurat_filt
# ============================================================
ht_clusters <- c("Neurons_4", "Neurons_10", "Neurons_16")

nd_seurat_filt$neuron_type <- ifelse(nd_seurat_filt$Subclass_new %in% ht_clusters,
                                     "5-HT", "non-5-HT-neurons")

cat("Neuron classification result:\n")
print(table(nd_seurat_filt$neuron_type))

# Visualize to confirm
DimPlot(nd_seurat_filt, group.by = "neuron_type", label = TRUE, repel = TRUE, raster = FALSE) +
  coord_fixed() + ggtitle("5-HT vs non-5-HT-neurons")

In [ ]:
# ============================================================
# Step 2: Write labels back to classFiltSeurat_filt
# ============================================================
classFiltSeurat_filt$cluster_final <- as.character(classFiltSeurat_filt$cluster)

# Update labels for Neuron cells
nd_cells_new <- colnames(nd_seurat_filt)
classFiltSeurat_filt$cluster_final[nd_cells_new] <- nd_seurat_filt$neuron_type

cat("\nFinal classification of all cells:\n")
print(table(classFiltSeurat_filt$cluster_final))



In [ ]:
# ============================================================
# Step 3: Extract 5-HT and non-5-HT neurons separately and re-cluster
# ============================================================

# -------- 3.1 Extract 5-HT neurons and cluster separately --------
ht_seurat <- subset(nd_seurat_filt, subset = neuron_type == "5-HT")
cat("\nNumber of 5-HT neurons:", ncol(ht_seurat), "\n")

# Recompute variable features + PCA + clustering + UMAP
ht_seurat <- FindVariableFeatures(ht_seurat, selection.method = "vst", nfeatures = 3000)
ht_seurat <- RunPCA(ht_seurat, features = VariableFeatures(ht_seurat), npcs = 50, verbose = FALSE)
ht_seurat <- FindNeighbors(ht_seurat, reduction = "pca", dims = 1:30)
ht_seurat <- FindClusters(ht_seurat, resolution = 0.5)  # Adjust as appropriate
ht_seurat <- RunUMAP(ht_seurat, reduction = "pca", dims = 1:30)

# Rename as 5-HT_0, 5-HT_1...
ht_clusters_new <- as.character(ht_seurat$seurat_clusters)
ht_unique <- sort(unique(as.numeric(ht_clusters_new)))

ht_counter <- 0
ht_map <- character(length(ht_unique))
names(ht_map) <- as.character(ht_unique)

for(cl in as.character(ht_unique)) {
  ht_map[cl] <- paste0("5-HT_", ht_counter)
  ht_counter <- ht_counter + 1
}

ht_seurat$ht_subtype <- unname(ht_map[ht_clusters_new])

cat("\n5-HT subtype distribution:\n")
print(table(ht_seurat$ht_subtype))

# 5-HT visualization
p_ht_umap <- DimPlot(ht_seurat, group.by = "ht_subtype", label = TRUE, repel = TRUE, raster = FALSE) +
  coord_fixed() + ggtitle("5-HT Neurons Subtypes")

p_ht_vln <- VlnPlot(ht_seurat, features = c("TPH2", "SLC6A4", "SLC17A6", "GAD1", "TH"),
                    pt.size = 0, ncol = 3)

print(p_ht_umap)
print(p_ht_vln)


In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)

# ============================================================
# 1. UMAP plot (by 5-HT subtypes)
# ============================================================
options(repr.plot.width = 10, repr.plot.height = 8)

p_ht_umap <- DimPlot(ht_seurat, group.by = "ht_subtype", 
                     label = TRUE, repel = TRUE, raster = FALSE) +
  coord_fixed() + 
  ggtitle("5-HT Neurons Subtypes") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"))

print(p_ht_umap)



In [ ]:
library(Seurat)
library(ggplot2)

# ============================================================
# 5-HT UMAP - no-label version
# ============================================================
options(repr.plot.width = 10, repr.plot.height = 8)

p_ht_umap_nolabel <- DimPlot(ht_seurat, 
                             group.by = "ht_subtype", 
                             label = FALSE,           # Turn off labels
                             raster = FALSE) +
  coord_fixed() + 
  ggtitle("5-HT Neurons Subtypes (No Label)") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"),
        legend.position = "right")  # Keep legend on the right for color identification

print(p_ht_umap_nolabel)

In [ ]:
# ============================================================
# 2. Find Top 20 marker genes
# ============================================================
ht_seurat <- SetIdent(ht_seurat, value = "ht_subtype")

ht_markers <- FindAllMarkers(ht_seurat, 
                             only.pos = TRUE, 
                             min.pct = 0.25, 
                             logfc.threshold = 0.25)

top20 <- ht_markers %>%
  group_by(cluster) %>%
  top_n(n = 20, wt = avg_log2FC) %>%
  arrange(cluster, desc(avg_log2FC))

cat("Top 20 markers per 5-HT subtype:\n")
print(top20 %>% select(gene, cluster, avg_log2FC, p_val_adj) %>% head(40))

write.csv(top20, "ht_top20_markers.csv", row.names = FALSE)


In [ ]:
library(Seurat)
library(ggplot2)
library(dplyr)
library(ranger)
library(pheatmap)
library(viridis)

# ============================================
# 0. Parameter settings
# ============================================
output_path <- "./data/"

# ============================================
# 1. Ensure 5-HT subset & fixed factor order
# ============================================
# Your data already has ht_subtype: 5-HT_0 ~ 5-HT_8
ht_seurat$ht_subtype <- factor(ht_seurat$ht_subtype, 
                                levels = paste0("5-HT_", 0:13))

# Set ident to ht_subtype for FindAllMarkers
ht_seurat <- SetIdent(ht_seurat, value = "ht_subtype")

cat("5-HT subtype distribution:\n")
print(table(ht_seurat$ht_subtype))

In [ ]:
# ============================================
# 2. Find marker genes (Wilcoxon)
# ============================================
ht_markers <- FindAllMarkers(ht_seurat,
                             only.pos = TRUE,
                             min.pct = 0.25,
                             logfc.threshold = 0.25,
                             test.use = "wilcox")

# Save Top 20
top20 <- ht_markers %>%
  group_by(cluster) %>%
  top_n(n = 20, wt = avg_log2FC) %>%
  arrange(cluster, desc(avg_log2FC))

write.csv(top20, paste0(output_path, "ht_top20_markers.csv"), row.names = FALSE)
cat("Top 20 markers saved\n")

# ============================================
# 3. Extract Top5 marker genes per subtype
# ============================================
top5_genes <- ht_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 5) %>%
  ungroup() %>%
  arrange(cluster)

cat("\nTop5 marker genes per subtype:\n")
print(top5_genes %>% select(cluster, gene, avg_log2FC, p_val_adj))

# Gene order (arranged by 5-HT_0 -> 5-HT_8, deduplicated)
gene_order <- top5_genes %>%
  arrange(cluster) %>%
  pull(gene) %>%
  unique()

cat("\nDot plot gene order (arranged 5-HT_0->8),", length(gene_order), "genes in total:\n")
cat(paste(gene_order, collapse = ", "), "\n")

# ============================================
# 4. Top5 DotPlot
# ============================================
p_dot <- DotPlot(
  ht_seurat,
  features = gene_order,
  group.by = "ht_subtype",         # Use ht_subtype
  dot.scale = 5,                   # Many genes, smaller dots
  cols = c("lightgrey", "red")
) +
  coord_flip() +
  RotatedAxis() +
  ggtitle("5-HT subtypes: Top5 marker gene dot plot") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 8),      # Many genes; shrink font to avoid overlap
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )

print(p_dot)

ggsave(
  filename = paste0(output_path, "5HT_Top5_Markers_DotPlot.png"),
  plot = p_dot,
  width = 14,      # 9 subtypes + 45 genes; wider canvas
  height = 16,     # Taller canvas
  dpi = 300
)
cat("\nTop5 dot plot saved!\n")

# Save table
write.csv(
  top5_genes %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
  file = paste0(output_path, "5HT_Top5_Markers_table.csv"),
  row.names = FALSE
)

In [ ]:
# ============================================
# 5. Violin plot: nFeature_RNA
# ============================================
plot_data <- ht_seurat@meta.data
plot_data$ht_subtype <- factor(plot_data$ht_subtype, levels = paste0("5-HT_", 0:13))

p_nfeature <- ggplot(plot_data, aes(x = ht_subtype, y = nFeature_RNA, fill = ht_subtype)) +
  geom_violin(scale = "width", trim = TRUE, width = 0.7, linewidth = 0.3, color = "black") +
  labs(x = "", y = "nFeature RNA") +
  theme_bw() +
  theme(
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    panel.border = element_blank(),
    axis.line = element_line(color = "black"),
    axis.ticks = element_line(color = "black"),
    axis.ticks.length = unit(6, "pt"),
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 10),
    axis.title = element_text(size = 13),
    legend.position = "none"
  )

print(p_nfeature)

png(paste0(output_path, "violin_5HT_nFeature_RNA.png"), width = 1200, height = 800, res = 150)
print(p_nfeature)
dev.off()

pdf(paste0(output_path, "violin_5HT_nFeature_RNA.pdf"), width = 12, height = 8)
print(p_nfeature)
dev.off()
cat("nFeature_RNA violin plot saved\n")

In [ ]:
# ============================================
# 6. Violin plot: nCount_RNA
# ============================================
p_ncount <- ggplot(plot_data, aes(x = ht_subtype, y = nCount_RNA, fill = ht_subtype)) +
  geom_violin(scale = "width", trim = TRUE, width = 0.7, linewidth = 0.3, color = "black") +
  labs(x = "", y = "nCount RNA") +
  theme_bw() +
  theme(
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    panel.border = element_blank(),
    axis.line = element_line(color = "black"),
    axis.ticks = element_line(color = "black"),
    axis.ticks.length = unit(6, "pt"),
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 10),
    axis.title = element_text(size = 13),
    legend.position = "none"
  )

print(p_ncount)

png(paste0(output_path, "violin_5HT_nCount_RNA.png"), width = 1200, height = 800, res = 150)
print(p_ncount)
dev.off()

pdf(paste0(output_path, "violin_5HT_nCount_RNA.pdf"), width = 12, height = 8)
print(p_ncount)
dev.off()
cat("nCount_RNA violin plot saved\n")

In [ ]:
# ============================================
# 7. Ranger + heatmap (values shown on diagonal only)
# ============================================
expr_matrix <- GetAssayData(ht_seurat, layer = "data")
genes <- VariableFeatures(ht_seurat)

X <- t(as.matrix(expr_matrix[genes, , drop = FALSE]))
y <- factor(ht_seurat$ht_subtype)

rf_ht <- ranger(
  x = X,
  y = y,
  num.trees = 500,
  num.threads = 8,
  classification = TRUE
)

cat("\nOOB Error:", rf_ht$prediction.error, "\n")

conf_mat <- rf_ht$confusion.matrix
confusionRatio <- t(apply(conf_mat, 1, function(x) {
  if(sum(x) == 0) return(rep(0, length(x)))
  x / sum(x)
}))

cat("Diagonal accuracy:\n")
print(round(diag(confusionRatio), 2))

# Filter subtypes with >= 20 cells
df_num <- as.data.frame(table(ht_seurat$ht_subtype))
select_cell <- as.character(df_num[df_num$Freq >= 20, ]$Var1)
select_cell <- intersect(select_cell, rownames(confusionRatio))

mat_to_plot <- confusionRatio[select_cell, select_cell, drop = FALSE]

# Show values on the diagonal only
num_labels <- matrix("", nrow = nrow(mat_to_plot), ncol = ncol(mat_to_plot))
diag(num_labels) <- sprintf("%.2f", diag(mat_to_plot))

# On-screen display
options(repr.plot.width = 10, repr.plot.height = 8)
pheatmap(
  mat_to_plot,
  breaks = seq(0, 1, 0.01),
  cluster_rows = FALSE,
  cluster_cols = FALSE,
  border_color = NA,
  cellwidth = 45,
  cellheight = 45,
  fontsize = 14,
  color = viridis(100),
  main = "5-HT Subtypes Ranger Confusion Matrix",
  display_numbers = num_labels,
  fontsize_number = 12,
  number_color = "white"
)

# Save PNG
png(paste0(output_path, "5HT_Ranger_Heatmap.png"), width = 1200, height = 1000, res = 150)
pheatmap(
  mat_to_plot,
  breaks = seq(0, 1, 0.01),
  cluster_rows = FALSE,
  cluster_cols = FALSE,
  border_color = NA,
  cellwidth = 45,
  cellheight = 45,
  fontsize = 14,
  color = viridis(100),
  main = "5-HT Subtypes Ranger Confusion Matrix",
  display_numbers = num_labels,
  fontsize_number = 12,
  number_color = "white"
)
dev.off()
cat("Ranger heatmaps saved\n")


In [ ]:
# library(Seurat)
# library(ggplot2)
# library(dplyr)
# library(ranger)
# library(pheatmap)
# library(viridis)

# # ============================================
# # 0. Parameter settings
# # ============================================
# output_path <- "/mnt/90-connectome/Personal/XQ/a-scientific-data/step5_output_829/"

# # ============================================
# # 1. Ensure 5-HT subset & fixed factor order (5-HT_0 ~ 5-HT_8)
# # ============================================
# ht_seurat$ht_subtype <- factor(ht_seurat$ht_subtype, levels = paste0("5-HT_", 0:8))
# ht_seurat <- SetIdent(ht_seurat, value = "ht_subtype")

# # ============================================
# # 2. Find marker genes (Wilcoxon) - skip if already run
# # ============================================
# ht_markers <- FindAllMarkers(ht_seurat,
#                              only.pos = TRUE,
#                              min.pct = 0.25,
#                              logfc.threshold = 0.25,
#                              test.use = "wilcox")

# # ============================================
# # 3. Extract Top10 marker genes per subtype
# # ============================================
# top10_genes <- ht_markers %>%
#   group_by(cluster) %>%
#   slice_max(order_by = avg_log2FC, n = 10) %>%
#   ungroup() %>%
#   arrange(cluster)

# cat("Top10 marker genes per subtype:\n")
# print(top10_genes %>% select(cluster, gene, avg_log2FC, p_val_adj))

# # Gene order (arranged by 5-HT_0 -> 5-HT_8, deduplicated)
# gene_order <- top10_genes %>%
#   arrange(cluster) %>%
#   pull(gene) %>%
#   unique()

# cat("\nDot plot gene order (arranged 5-HT_0->8),", length(gene_order), "genes in total:\n")

# # ============================================
# # 4. Top10 DotPlot
# # ============================================
# p_dot10 <- DotPlot(
#   ht_seurat,
#   features = gene_order,
#   group.by = "ht_subtype",
#   dot.scale = 4,                   # Many genes, smaller dots
#   cols = c("lightgrey", "red")
# ) +
#   coord_flip() +
#   RotatedAxis() +
#   ggtitle("5-HT subtypes: Top10 marker gene dot plot") +
#   theme(
#     axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
#     axis.text.y = element_text(size = 7),      # ~90 genes; even smaller font
#     plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
#   )

# print(p_dot10)

# ggsave(
#   filename = paste0(output_path, "5HT_Top10_Markers_DotPlot.png"),
#   plot = p_dot10,
#   width = 16,      # 9 subtypes; wider canvas
#   height = 22,     # ~90 genes; very tall canvas
#   dpi = 300
# )
# cat("Top10 dot plot saved!\n")

# # Save table
# write.csv(
#   top10_genes %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
#   file = paste0(output_path, "5HT_Top10_Markers_table.csv"),
#   row.names = FALSE
# )

# # ============================================
# # 5. Violin plot: nFeature_RNA (ticks every 1000)
# # ============================================
# plot_data <- ht_seurat@meta.data
# plot_data$ht_subtype <- factor(plot_data$ht_subtype, levels = paste0("5-HT_", 0:8))

# # Compute appropriate tick range
# nfeature_max <- max(plot_data$nFeature_RNA, na.rm = TRUE)
# nfeature_breaks <- seq(0, ceiling(nfeature_max / 1000) * 1000, by = 1000)

# p_nfeature <- ggplot(plot_data, aes(x = ht_subtype, y = nFeature_RNA, fill = ht_subtype)) +
#   geom_violin(scale = "width", trim = TRUE, width = 0.7, linewidth = 0.3, color = "black") +
#   scale_y_continuous(breaks = nfeature_breaks) +   # Tick marks every 1000
#   labs(x = "", y = "nFeature RNA") +
#   theme_bw() +
#   theme(
#     panel.grid.major = element_blank(),
#     panel.grid.minor = element_blank(),
#     panel.border = element_blank(),
#     axis.line = element_line(color = "black"),
#     axis.ticks = element_line(color = "black"),
#     axis.ticks.length = unit(6, "pt"),
#     axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
#     axis.text.y = element_text(size = 10),
#     axis.title = element_text(size = 13),
#     legend.position = "none"
#   )

# print(p_nfeature)

# png(paste0(output_path, "violin_5HT_nFeature_RNA_1k.png"), width = 1200, height = 800, res = 150)
# print(p_nfeature)
# dev.off()

# pdf(paste0(output_path, "violin_5HT_nFeature_RNA_1k.pdf"), width = 12, height = 8)
# print(p_nfeature)
# dev.off()
# cat("nFeature_RNA violin plot (1000 ticks) saved\n")

# # ============================================
# # 6. Violin plot: nCount_RNA
# # ============================================
# p_ncount <- ggplot(plot_data, aes(x = ht_subtype, y = nCount_RNA, fill = ht_subtype)) +
#   geom_violin(scale = "width", trim = TRUE, width = 0.7, linewidth = 0.3, color = "black") +
#   labs(x = "", y = "nCount RNA") +
#   theme_bw() +
#   theme(
#     panel.grid.major = element_blank(),
#     panel.grid.minor = element_blank(),
#     panel.border = element_blank(),
#     axis.line = element_line(color = "black"),
#     axis.ticks = element_line(color = "black"),
#     axis.ticks.length = unit(6, "pt"),
#     axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
#     axis.text.y = element_text(size = 10),
#     axis.title = element_text(size = 13),
#     legend.position = "none"
#   )

# print(p_ncount)

# png(paste0(output_path, "violin_5HT_nCount_RNA.png"), width = 1200, height = 800, res = 150)
# print(p_ncount)
# dev.off()

# pdf(paste0(output_path, "violin_5HT_nCount_RNA.pdf"), width = 12, height = 8)
# print(p_ncount)
# dev.off()
# cat("nCount_RNA violin plot saved\n")

# # ============================================
# # 7. Ranger + heatmap (no-values version)
# # ============================================
# expr_matrix <- GetAssayData(ht_seurat, layer = "data")
# genes <- VariableFeatures(ht_seurat)

# X <- t(as.matrix(expr_matrix[genes, , drop = FALSE]))
# y <- factor(ht_seurat$ht_subtype)

# rf_ht <- ranger(
#   x = X,
#   y = y,
#   num.trees = 500,
#   num.threads = 8,
#   classification = TRUE
# )

# cat("\nOOB Error:", rf_ht$prediction.error, "\n")

# conf_mat <- rf_ht$confusion.matrix
# confusionRatio <- t(apply(conf_mat, 1, function(x) {
#   if(sum(x) == 0) return(rep(0, length(x)))
#   x / sum(x)
# }))

# cat("Diagonal accuracy:\n")
# print(round(diag(confusionRatio), 2))

# # Filter subtypes with >= 20 cells
# df_num <- as.data.frame(table(ht_seurat$ht_subtype))
# select_cell <- as.character(df_num[df_num$Freq >= 20, ]$Var1)
# select_cell <- intersect(select_cell, rownames(confusionRatio))

# mat_to_plot <- confusionRatio[select_cell, select_cell, drop = FALSE]

# # ---------- 7.1 With-values version (kept) ----------
# num_labels <- matrix("", nrow = nrow(mat_to_plot), ncol = ncol(mat_to_plot))
# diag(num_labels) <- sprintf("%.2f", diag(mat_to_plot))

# png(paste0(output_path, "5HT_Ranger_Heatmap_withNum.png"), width = 1200, height = 1000, res = 150)
# pheatmap(
#   mat_to_plot,
#   breaks = seq(0, 1, 0.01),
#   cluster_rows = FALSE,
#   cluster_cols = FALSE,
#   border_color = NA,
#   cellwidth = 45,
#   cellheight = 45,
#   fontsize = 14,
#   color = viridis(100),
#   main = "5-HT Subtypes Ranger Confusion Matrix",
#   display_numbers = num_labels,
#   fontsize_number = 12,
#   number_color = "white"
# )
# dev.off()
# cat("Ranger heatmap (with values) saved\n")

# # ---------- 7.2 No-values version (newly added) ----------
# png(paste0(output_path, "5HT_Ranger_Heatmap_noNum.png"), width = 1200, height = 1000, res = 150)
# pheatmap(
#   mat_to_plot,
#   breaks = seq(0, 1, 0.01),
#   cluster_rows = FALSE,
#   cluster_cols = FALSE,
#   border_color = NA,
#   cellwidth = 45,
#   cellheight = 45,
#   fontsize = 14,
#   color = viridis(100),
#   main = "5-HT Subtypes Ranger Confusion Matrix",
#   display_numbers = FALSE      # Values hidden
# )
# dev.off()
# cat("Ranger heatmap (no values) saved\n")

# # On-screen display, no-values version
# pheatmap(
#   mat_to_plot,
#   breaks = seq(0, 1, 0.01),
#   cluster_rows = FALSE,
#   cluster_cols = FALSE,
#   border_color = NA,
#   cellwidth = 45,
#   cellheight = 45,
#   fontsize = 14,
#   color = viridis(100),
#   main = "5-HT Subtypes Ranger Confusion Matrix (No Numbers)",
#   display_numbers = FALSE
# )

# cat("\n===== All done =====\n")


In [ ]:
# library(Seurat)
# library(ggplot2)
# library(dplyr)

# output_path <- "/mnt/90-connectome/Personal/XQ/a-scientific-data/step5_output_829/"

# # ============================================
# # 1. Ensure factor order & ident setting
# # ============================================
# ht_seurat$ht_subtype <- factor(ht_seurat$ht_subtype, levels = paste0("5-HT_", 0:8))
# ht_seurat <- SetIdent(ht_seurat, value = "ht_subtype")

# # ============================================
# # 2. Extract Top20 marker genes per subtype
# # ============================================
# top20_genes <- ht_markers %>%
#   group_by(cluster) %>%
#   slice_max(order_by = avg_log2FC, n = 20) %>%
#   ungroup() %>%
#   arrange(cluster)

# cat("Top20 marker genes per subtype:\n")
# print(top20_genes %>% select(cluster, gene, avg_log2FC, p_val_adj))

# # Gene order (arranged by 5-HT_0 -> 5-HT_8, deduplicated)
# gene_order <- top20_genes %>%
#   arrange(cluster) %>%
#   pull(gene) %>%
#   unique()

# cat("\nTotal dot plot genes:", length(gene_order), "\n")

# # ============================================
# # 3. Top20 DotPlot
# # ============================================
# p_dot20 <- DotPlot(
#   ht_seurat,
#   features = gene_order,
#   group.by = "ht_subtype",
#   dot.scale = 3,                   # Very many genes, small dots
#   cols = c("lightgrey", "red")
# ) +
#   coord_flip() +
#   RotatedAxis() +
#   ggtitle("5-HT subtypes: Top20 marker gene dot plot") +
#   theme(
#     axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
#     axis.text.y = element_text(size = 5.5),     # ~180 genes; font must be very small
#     plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
#     panel.grid.major = element_blank(),
#     panel.grid.minor = element_blank()
#   )

# print(p_dot20)

# ggsave(
#   filename = paste0(output_path, "5HT_Top20_Markers_DotPlot.png"),
#   plot = p_dot20,
#   width = 18,      # 9 subtypes; very wide x-axis
#   height = 32,     # ~180 genes; very tall y-axis
#   dpi = 300
# )
# cat("Top20 dot plot saved!\n")

# # Save table
# write.csv(
#   top20_genes %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
#   file = paste0(output_path, "5HT_Top20_Markers_table.csv"),
#   row.names = FALSE
# )
# cat("Top20 marker gene table saved!\n")

In [ ]:
# library(Seurat)
# library(ggplot2)

# output_path <- "/mnt/90-connectome/Personal/XQ/a-scientific-data/step5_output_829/"

# # ============================================
# # 1. Ensure factor order
# # ============================================
# ht_seurat$ht_subtype <- factor(ht_seurat$ht_subtype, levels = paste0("5-HT_", 0:8))

# # ============================================
# # 2. Specified gene list (in your 9-group order)
# # ============================================
# gene_list <- c(
#   "ARPP21", "TLL1", "RORB",
#   "CHN2", "TMTC1", "KHDRBS2",
#   "PLPP4", "ITPR2", "NRP2",
#   "LDB2", "CACNG3", "ZNF536",
#   "ABI3BP", "CDH7", "KIAA1456",
#   "FRAS1", "CRYBG3", "GRM1",
#   "PTPRT", "PDLIM5", "TSHZ2",
#   "ARHGAP15", "ZEB2", "CALCR",
#   "CNTN5", "UNC13C", "ARHGEF3"
# )

# # Check which genes are present in the data
# available_genes <- gene_list[gene_list %in% rownames(ht_seurat)]
# missing_genes <- gene_list[!gene_list %in% rownames(ht_seurat)]

# if(length(missing_genes) > 0) {
#   cat("WARNING: the following genes were not found in the expression matrix and will be skipped:\n")
#   cat(paste(missing_genes, collapse = ", "), "\n\n")
# }
# cat("OK: available genes (", length(available_genes), "):\n")
# cat(paste(available_genes, collapse = ", "), "\n\n")

# # ============================================
# # 3. Draw DotPlot
# # ============================================
# # coord_flip(): X axis = cell types (5-HT_0~8), Y axis = genes
# p_custom <- DotPlot(
#   ht_seurat,
#   features = available_genes,
#   group.by = "ht_subtype",
#   dot.scale = 5,
#   cols = c("lightgrey", "red")
# ) +
#   coord_flip() +
#   RotatedAxis() +
#   ggtitle("5-HT subtypes: specified marker gene dot plot") +
#   xlab("5-HT subtype") +
#   ylab("Gene") +
#   theme(
#     axis.text.x = element_text(angle = 45, hjust = 1, size = 10, face = "bold"),
#     axis.text.y = element_text(size = 10),
#     axis.title = element_text(size = 12, face = "bold"),
#     plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
#     panel.grid.major = element_blank(),
#     panel.grid.minor = element_blank()
#   )

# print(p_custom)

# ggsave(
#   filename = paste0(output_path, "5HT_Custom_Markers_DotPlot.png"),
#   plot = p_custom,
#   width = 12,
#   height = 10,
#   dpi = 300
# )
# cat("Specified-gene dot plot saved!\n")

In [ ]:
# Check all column names first
colnames(classFiltSeurat@meta.data)

In [ ]:
library(Seurat)

# ============================================
# Step 5: Write refined 5-HT subtypes back to the original object
# ============================================

# Use existing cluster column as default
classFiltSeurat$cell_type <- as.character(classFiltSeurat$cluster)

# 5-HT cells: overwrite with refined subtypes (5-HT_0 ~ 5-HT_8)
ht_cells <- colnames(ht_seurat)
classFiltSeurat$cell_type[ht_cells] <- as.character(ht_seurat$ht_subtype)

# Fix factor order
all_types <- unique(classFiltSeurat$cell_type)
ht_types <- paste0("5-HT_", 0:8)
other_types <- setdiff(all_types, ht_types)
type_levels <- c(ht_types, sort(other_types))

classFiltSeurat$cell_type <- factor(classFiltSeurat$cell_type, levels = type_levels)

cat("\ncell_type distribution of all cells:\n")
print(table(classFiltSeurat$cell_type))

In [ ]:
save_path <- "./data/"
dir.create(save_path, showWarnings = FALSE, recursive = TRUE)

# 1. Full object RDS (contains cell_type column)
saveRDS(classFiltSeurat,
        file = paste0(save_path, "classFiltSeurat_with_ht_subtype.rds"))
cat("OK: full Seurat object saved!\n")

# 2. Metadata CSV
write.csv(classFiltSeurat@meta.data,
          file = paste0(save_path, "classFiltSeurat_metadata.csv"),
          row.names = TRUE)
cat("OK: metadata saved!\n")

# 3. 5-HT subset RDS
saveRDS(ht_seurat,
        file = paste0(save_path, "ht_seurat_harmony_base.rds"))
cat("OK: 5-HT subset saved!\n")

In [ ]:
# Preliminary 5-HT classification above, to avoid 5-HT contamination in non-5-HT neurons later

In [ ]:
# Extract non-5-HT neurons and re-cluster separately

In [ ]:
library(Seurat)
save_path <- "./data/"

# ---- 1. Read back both objects ----
classFiltSeurat <- readRDS(paste0(save_path, "classFiltSeurat_with_ht_subtype.rds"))
ht_seurat       <- readRDS(paste0(save_path, "ht_seurat_harmony_base.rds"))

ht_cells <- intersect(colnames(ht_seurat), colnames(classFiltSeurat))
stopifnot(length(ht_cells) == 3844)

# ---- 2. Rebuild labels (revised version; the change is here) ----
cl        <- as.character(classFiltSeurat$cluster)
is_neuron <- startsWith(cl, "Neurons")     # The original == "Neurons" matched no cells; replaced with startsWith
all_bc    <- colnames(classFiltSeurat)

classFiltSeurat$cluster_final <- cl
classFiltSeurat$cluster_final[all_bc %in% ht_cells] <- "5-HT"
classFiltSeurat$cluster_final[is_neuron & !(all_bc %in% ht_cells)] <- "non-5-HT-neurons"

# ---- 3. View results ----
final_table <- table(classFiltSeurat$cluster_final)
print(final_table)


In [ ]:
cf <- classFiltSeurat$cluster_final
nf <- classFiltSeurat$nFeature_RNA

print(addmargins(table(cf, nf > 1500)))   # Per-class counts before/after filtering

cat("non-5-HT with nFeature>1500:", sum(cf == "non-5-HT-neurons" & nf > 1500), "\n")
cat("5-HT with nFeature>1500:",     sum(cf == "5-HT" & nf > 1500), "\n")

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(harmony)

# ============================================================
# Step 3.1: Extract non-5-HT neurons (with nFeature > 1500 filter)
# ============================================================
non_ht_seurat <- subset(classFiltSeurat,
                        subset = cluster_final == "non-5-HT-neurons" &
                                 nFeature_RNA > 1500)

cat("non-5-HT neurons (after extraction + nFeature>1500):", ncol(non_ht_seurat), "\n")
stopifnot(ncol(non_ht_seurat) == 41209)

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(harmony)
library(clustree)   # Newly added

# ============================================================
# Step 3.2: Normalization + Harmony (unchanged)
# ============================================================
non_ht_seurat <- SCTransform(non_ht_seurat,
                             vst.flavor = "v2",
                             variable.features.n = 3000,
                             vars.to.regress = "percent.mt",
                             method = "glmGamPoi",
                             verbose = FALSE)

non_ht_seurat <- RunPCA(non_ht_seurat, npcs = 50, verbose = FALSE)

non_ht_seurat <- RunHarmony(non_ht_seurat,
                            group.by.vars = "donor",
                            reduction.use = "pca",
                            dims.use = 1:30,
                            assay.use = "SCT",
                            verbose = FALSE)

non_ht_seurat <- FindNeighbors(non_ht_seurat, reduction = "harmony", dims = 1:30)

# ============================================================
# New: multi-resolution clustering + clustree
# ============================================================
# Note: no resolution set here; defaults to 0.8
# Then run a series of resolutions manually; results stored in meta.data
resolutions <- seq(0.1, 1.0, by = 0.1)

for (res in resolutions) {
  non_ht_seurat <- FindClusters(non_ht_seurat,
                                resolution = res,
                                future.seed = TRUE,
                                verbose = FALSE)
  cat("Completed resolution =", res, "\n")
}

# clustree visualization (column prefix is "SCT_snn_res.")
p_clustree <- clustree(non_ht_seurat@meta.data,
                       prefix = "SCT_snn_res.",
                       node_colour = "sc3_stability") +
  ggtitle("Clustree: non-5-HT neurons (res 0.1-1.0)") +
  theme(legend.position = "right")
print(p_clustree)

ggsave("./data/nonHT_clustree.png",
       p_clustree, width = 12, height = 8, dpi = 300)

In [ ]:


# ============================================================
# Step 3.2: Normalization + Harmony + clustering
# ============================================================
non_ht_seurat <- SCTransform(non_ht_seurat,
                             vst.flavor = "v2",
                             variable.features.n = 3000,
                             vars.to.regress = "percent.mt",
                             method = "glmGamPoi",
                             verbose = FALSE)

non_ht_seurat <- RunPCA(non_ht_seurat, npcs = 50, verbose = FALSE)

non_ht_seurat <- RunHarmony(non_ht_seurat,
                            group.by.vars = "donor",
                            reduction.use = "pca",
                            dims.use = 1:30,
                            assay.use = "SCT",
                            verbose = FALSE)

non_ht_seurat <- FindNeighbors(non_ht_seurat, reduction = "harmony", dims = 1:30)
non_ht_seurat <- FindClusters(non_ht_seurat, resolution = 0.3, future.seed = TRUE)
non_ht_seurat <- RunUMAP(non_ht_seurat, reduction = "harmony", dims = 1:30)

# ---------- Rename as nonHT_0, nonHT_1... ----------
non_ht_clusters_new <- as.character(non_ht_seurat$seurat_clusters)
non_ht_unique <- sort(unique(as.numeric(non_ht_clusters_new)))

non_ht_counter <- 0
non_ht_map <- character(length(non_ht_unique))
names(non_ht_map) <- as.character(non_ht_unique)

for (cl in as.character(non_ht_unique)) {
  non_ht_map[cl] <- paste0("nonHT_", non_ht_counter)
  non_ht_counter <- non_ht_counter + 1
}

non_ht_seurat$non_ht_subtype <- unname(non_ht_map[non_ht_clusters_new])

cat("\nnon-5-HT subtype distribution (res 0.3, n=41209):\n")
print(table(non_ht_seurat$non_ht_subtype))

# ---------- QC visualization ----------
p1 <- DimPlot(non_ht_seurat, group.by = "donor", raster = FALSE) +
  ggtitle("non-5-HT neurons by donor (n=41209, nFeature>1500, harmony)") +
  coord_fixed()
print(p1)

p2 <- DimPlot(non_ht_seurat, group.by = "non_ht_subtype",
              label = TRUE, repel = TRUE, raster = FALSE) +
  ggtitle("non-5-HT neurons reclustered (res 0.3, n=41209)") +
  coord_fixed()
print(p2)

p3 <- DotPlot(non_ht_seurat,
              features = c("SNAP25", "SLC17A6", "SLC17A8", "GAD1", "GAD2", "TH"),
              group.by = "non_ht_subtype") +
  RotatedAxis()
print(p3)




In [ ]:
# ---------- Archive immediately after successful run (do not keep in memory only) ----------
saveRDS(non_ht_seurat,
        file = paste0(save_path, "non_ht_seurat_final.rds"))
saveRDS(classFiltSeurat,
        file = paste0(save_path, "classFiltSeurat_final.rds"))
cat("OK: non_ht_seurat and classFiltSeurat saved to disk\n")

In [ ]:
library(Seurat)
library(ggplot2)
library(dplyr)
library(ranger)
library(pheatmap)
library(viridis)

# ============================================
# 0. Parameter settings
# ============================================
output_path <- "./data/"

# ============================================
# 1. Data preparation: ensure factor order & ident
# ============================================
# Dynamically obtain and sort non-HT subtype levels
non_ht_levels <- sort(unique(as.character(non_ht_seurat$non_ht_subtype)))
non_ht_seurat$non_ht_subtype <- factor(non_ht_seurat$non_ht_subtype, levels = non_ht_levels)
non_ht_seurat <- SetIdent(non_ht_seurat, value = "non_ht_subtype")

cat("non-5-HT subtype distribution:\n")
print(table(non_ht_seurat$non_ht_subtype))

In [ ]:
# ============================================
# 1. Data preparation: ensure factor order & ident (numeric order 0,1,2...)
# ============================================
# First get all unique subtype names
non_ht_levels <- unique(as.character(non_ht_seurat$non_ht_subtype))

# Extract numeric part and sort numerically (ensure nonHT_0,1,2... not lexicographic order)
non_ht_nums <- as.numeric(gsub("nonHT_", "", non_ht_levels))
non_ht_levels <- non_ht_levels[order(non_ht_nums)]

# Or in one line:
# non_ht_levels <- paste0("nonHT_", sort(as.numeric(gsub("nonHT_", "", unique(as.character(non_ht_seurat$non_ht_subtype))))))

non_ht_seurat$non_ht_subtype <- factor(non_ht_seurat$non_ht_subtype, levels = non_ht_levels)
non_ht_seurat <- SetIdent(non_ht_seurat, value = "non_ht_subtype")

cat("non-5-HT subtype distribution:\n")
print(table(non_ht_seurat$non_ht_subtype))

In [ ]:
# ============================================
# 2. UMAP plots
# ============================================
# 2.1 Labeled version
p_nonht_umap <- DimPlot(non_ht_seurat, group.by = "non_ht_subtype", 
                        label = TRUE, repel = TRUE, raster = FALSE) +
  coord_fixed() + 
  ggtitle("non-5-HT Neurons Subtypes") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"))

print(p_nonht_umap)
ggsave(paste0(output_path, "nonHT_UMAP_labeled.png"), p_nonht_umap, width = 10, height = 8, dpi = 300)

# 2.2 No-label version
p_nonht_umap_nolabel <- DimPlot(non_ht_seurat, group.by = "non_ht_subtype", 
                                label = FALSE, raster = FALSE) +
  coord_fixed() + 
  ggtitle("non-5-HT Neurons Subtypes (No Label)") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"),
        legend.position = "right")

print(p_nonht_umap_nolabel)
ggsave(paste0(output_path, "nonHT_UMAP_nolabel.png"), p_nonht_umap_nolabel, width = 10, height = 8, dpi = 300)

cat("UMAP plots saved\n")


In [ ]:
# ============================================
# 3. Find marker genes (Wilcoxon) - run only once
# ============================================
nonht_markers <- FindAllMarkers(non_ht_seurat,
                                only.pos = TRUE,
                                min.pct = 0.25,
                                logfc.threshold = 0.25,
                                test.use = "wilcox")

# Save Top20
top20 <- nonht_markers %>%
  group_by(cluster) %>%
  top_n(n = 20, wt = avg_log2FC) %>%
  arrange(cluster, desc(avg_log2FC))

write.csv(top20, paste0(output_path, "nonHT_top20_markers.csv"), row.names = FALSE)
cat("Top20 markers saved\n")

# ============================================
# 4. Top5 DotPlot
# ============================================
top5_genes <- nonht_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 5) %>%
  ungroup() %>%
  arrange(cluster)

gene_order5 <- top5_genes %>% arrange(cluster) %>% pull(gene) %>% unique()

p_dot5 <- DotPlot(non_ht_seurat, features = gene_order5, group.by = "non_ht_subtype",
                  dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("non-5-HT subtypes: Top5 marker gene dot plot") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 8),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )

print(p_dot5)
ggsave(paste0(output_path, "nonHT_Top5_Markers_DotPlot.png"), p_dot5, 
       width = 14, height = 16, dpi = 300)

write.csv(top5_genes %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "nonHT_Top5_Markers_table.csv"), row.names = FALSE)
cat("Top5 dot plot saved\n")

# ============================================
# 5. Top10 DotPlot
# ============================================
top10_genes <- nonht_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 10) %>%
  ungroup() %>%
  arrange(cluster)

gene_order10 <- top10_genes %>% arrange(cluster) %>% pull(gene) %>% unique()

p_dot10 <- DotPlot(non_ht_seurat, features = gene_order10, group.by = "non_ht_subtype",
                   dot.scale = 4, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("non-5-HT subtypes: Top10 marker gene dot plot") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 7),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )

print(p_dot10)
ggsave(paste0(output_path, "nonHT_Top10_Markers_DotPlot.png"), p_dot10, 
       width = 16, height = 22, dpi = 300)

write.csv(top10_genes %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "nonHT_Top10_Markers_table.csv"), row.names = FALSE)
cat("Top10 dot plot saved\n")

# ============================================
# 6. Top20 DotPlot
# ============================================
gene_order20 <- top20 %>% arrange(cluster) %>% pull(gene) %>% unique()

p_dot20 <- DotPlot(non_ht_seurat, features = gene_order20, group.by = "non_ht_subtype",
                   dot.scale = 3, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("non-5-HT subtypes: Top20 marker gene dot plot") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 5.5),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )

print(p_dot20)
ggsave(paste0(output_path, "nonHT_Top20_Markers_DotPlot.png"), p_dot20, 
       width = 18, height = 32, dpi = 300)

write.csv(top20 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "nonHT_Top20_Markers_table.csv"), row.names = FALSE)
cat("Top20 dot plot saved\n")

In [ ]:
library(Seurat)
library(ggplot2)
library(dplyr)
library(ranger)
library(pheatmap)
library(viridis)

output_path <- "/mnt/90-connectome/Personal/XQ/a-nc-raphe/step5_output_1019/"

# ============================================
# 0. Data preparation (using the cleaned object)
# ============================================
# Re-obtain and fix factor levels (sorted numerically 0,1,2...)
nonht_levels <- paste0("nonHT_", sort(as.numeric(gsub("nonHT_", "", 
                unique(as.character(non_ht_seurat$non_ht_subtype))))))
non_ht_seurat$non_ht_subtype <- factor(non_ht_seurat$non_ht_subtype, levels = nonht_levels)
non_ht_seurat_clean <- SetIdent(non_ht_seurat, value = "non_ht_subtype")

cat("non-5-HT subtype distribution:\n")
print(table(non_ht_seurat_clean$non_ht_subtype))


In [ ]:
# ============================================
# 1. UMAP plots
# ============================================
# 1.1 Labeled version
p_nonht_umap <- DimPlot(non_ht_seurat_clean, group.by = "non_ht_subtype", 
                        label = TRUE, repel = TRUE, raster = FALSE) +
  coord_fixed() + 
  ggtitle("non-5-HT Neurons Subtypes") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"))
print(p_nonht_umap)
ggsave(paste0(output_path, "nonHT_UMAP_labeled.png"), p_nonht_umap, width = 10, height = 8, dpi = 300)

# 1.2 No-label version
p_nonht_umap_nolabel <- DimPlot(non_ht_seurat_clean, group.by = "non_ht_subtype", 
                                label = FALSE, raster = FALSE) +
  coord_fixed() + 
  ggtitle("non-5-HT Neurons Subtypes (No Label)") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"),
        legend.position = "right")
print(p_nonht_umap_nolabel)
ggsave(paste0(output_path, "nonHT_UMAP_nolabel.png"), p_nonht_umap_nolabel, width = 10, height = 8, dpi = 300)
cat("UMAP plots saved\n")


In [ ]:
# ============================================
# 6. Specified-gene DotPlot + UMAP FeaturePlot
# ============================================
# Specified genes; auto-correct case; ChE matched to ACHE
gene_list_input <- c('SLC17A6','SLC17A7','SLC17A8','GAD1','GAD2','TH')
gene_list <- toupper(gene_list_input)  # Unified to uppercase
gene_list[gene_list == "CHE"] <- "ACHE"  # ChE usually refers to ACHE (acetylcholinesterase)

available_genes <- gene_list[gene_list %in% rownames(non_ht_seurat_clean)]
missing_genes <- gene_list[!gene_list %in% rownames(non_ht_seurat_clean)]

if(length(missing_genes) > 0) {
  cat("WARNING: the following genes were not found in the expression matrix and will be skipped:\n")
  cat(paste(missing_genes, collapse = ", "), "\n\n")
}
cat("OK: available genes (", length(available_genes), "):\n")
cat(paste(available_genes, collapse = ", "), "\n\n")

# 6.1 DotPlot
p_custom <- DotPlot(non_ht_seurat_clean, features = available_genes, group.by = "non_ht_subtype",
                    dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("non-5-HT subtypes: specified marker gene dot plot") +
  xlab("non-5-HT subtype") + ylab("Gene") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 10, face = "bold"),
    axis.text.y = element_text(size = 10),
    axis.title = element_text(size = 12, face = "bold"),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )
print(p_custom)
ggsave(paste0(output_path, "nonHT_Custom_Markers_DotPlot.png"), p_custom, 
       width = 12, height = 10, dpi = 300)
cat("Specified-gene dot plot saved\n")

# 6.2 UMAP FeaturePlot (faceted)
p_features <- FeaturePlot(non_ht_seurat_clean, features = available_genes, 
                          ncol = 4, raster = FALSE, cols = c("lightgrey", "red"))
print(p_features)
ggsave(paste0(output_path, "nonHT_Custom_Markers_UMAP.png"), p_features, 
       width = 16, height = 12, dpi = 300)
cat("Specified-gene UMAP FeaturePlot saved\n")

In [ ]:
# ============================================
# 6. Specified-gene DotPlot + UMAP FeaturePlot
# ============================================
# Specified genes; auto-correct case; ChE matched to ACHE
gene_list_input <- c('SLC17A6','SLC17A8','GAD1','GAD2','SOX5','GFAP','PECAM1','FLT1','VCAN','MOBP','C1QC','SLC6A4','TH')
gene_list <- toupper(gene_list_input)  # Unified to uppercase
gene_list[gene_list == "CHE"] <- "ACHE"  # ChE usually refers to ACHE (acetylcholinesterase)

available_genes <- gene_list[gene_list %in% rownames(non_ht_seurat_clean)]
missing_genes <- gene_list[!gene_list %in% rownames(non_ht_seurat_clean)]

if(length(missing_genes) > 0) {
  cat("WARNING: the following genes were not found in the expression matrix and will be skipped:\n")
  cat(paste(missing_genes, collapse = ", "), "\n\n")
}
cat("OK: available genes (", length(available_genes), "):\n")
cat(paste(available_genes, collapse = ", "), "\n\n")

# 6.1 DotPlot
p_custom <- DotPlot(non_ht_seurat_clean, features = available_genes, group.by = "non_ht_subtype",
                    dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("non-5-HT subtypes: specified marker gene dot plot") +
  xlab("non-5-HT subtype") + ylab("Gene") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 10, face = "bold"),
    axis.text.y = element_text(size = 10),
    axis.title = element_text(size = 12, face = "bold"),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )
print(p_custom)
ggsave(paste0(output_path, "nonHT_Custom_Markers_DotPlot.png"), p_custom, 
       width = 12, height = 10, dpi = 300)
cat("Specified-gene dot plot saved\n")

# 6.2 UMAP FeaturePlot (faceted)
p_features <- FeaturePlot(non_ht_seurat_clean, features = available_genes, 
                          ncol = 4, raster = FALSE, cols = c("lightgrey", "red"))
print(p_features)
ggsave(paste0(output_path, "nonHT_Custom_Markers_UMAP.png"), p_features, 
       width = 16, height = 12, dpi = 300)
cat("Specified-gene UMAP FeaturePlot saved\n")

In [ ]:
# ============================================
# 7. Violin plots
# ============================================
plot_data <- non_ht_seurat_clean@meta.data
plot_data$non_ht_subtype <- factor(plot_data$non_ht_subtype, levels = nonht_levels)

# 7.1 nFeature_RNA (ticks every 1000)
nfeature_max <- max(plot_data$nFeature_RNA, na.rm = TRUE)
nfeature_breaks <- seq(0, ceiling(nfeature_max / 1000) * 1000, by = 1000)

p_nfeature <- ggplot(plot_data, aes(x = non_ht_subtype, y = nFeature_RNA, fill = non_ht_subtype)) +
  geom_violin(scale = "width", trim = TRUE, width = 0.7, linewidth = 0.3, color = "black") +
  scale_y_continuous(breaks = nfeature_breaks) +
  labs(x = "", y = "nFeature RNA") +
  theme_bw() +
  theme(
    panel.grid.major = element_blank(), panel.grid.minor = element_blank(),
    panel.border = element_blank(), axis.line = element_line(color = "black"),
    axis.ticks = element_line(color = "black"), axis.ticks.length = unit(6, "pt"),
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 10), axis.title = element_text(size = 13),
    legend.position = "none"
  )
print(p_nfeature)
ggsave(paste0(output_path, "violin_nonHT_nFeature_RNA_1k.png"), p_nfeature, width = 8, height = 6, dpi = 150)
ggsave(paste0(output_path, "violin_nonHT_nFeature_RNA_1k.pdf"), p_nfeature, width = 12, height = 8)

# 7.2 nCount_RNA
p_ncount <- ggplot(plot_data, aes(x = non_ht_subtype, y = nCount_RNA, fill = non_ht_subtype)) +
  geom_violin(scale = "width", trim = TRUE, width = 0.7, linewidth = 0.3, color = "black") +
  labs(x = "", y = "nCount RNA") +
  theme_bw() +
  theme(
    panel.grid.major = element_blank(), panel.grid.minor = element_blank(),
    panel.border = element_blank(), axis.line = element_line(color = "black"),
    axis.ticks = element_line(color = "black"), axis.ticks.length = unit(6, "pt"),
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 10), axis.title = element_text(size = 13),
    legend.position = "none"
  )
print(p_ncount)
ggsave(paste0(output_path, "violin_nonHT_nCount_RNA.png"), p_ncount, width = 8, height = 6, dpi = 150)
ggsave(paste0(output_path, "violin_nonHT_nCount_RNA.pdf"), p_ncount, width = 12, height = 8)
cat("Violin plots saved\n")


In [ ]:
# ============================================
# 8. Ranger + heatmap
# ============================================
expr_matrix <- GetAssayData(non_ht_seurat_clean, layer = "data")
genes <- VariableFeatures(non_ht_seurat_clean)

X <- t(as.matrix(expr_matrix[genes, , drop = FALSE]))
y <- factor(non_ht_seurat_clean$non_ht_subtype)

rf_nonht <- ranger(x = X, y = y, num.trees = 500, num.threads = 8, classification = TRUE)

cat("\nOOB Error:", rf_nonht$prediction.error, "\n")

conf_mat <- rf_nonht$confusion.matrix
confusionRatio <- t(apply(conf_mat, 1, function(x) {
  if(sum(x) == 0) return(rep(0, length(x)))
  x / sum(x)
}))

cat("Diagonal accuracy:\n")
print(round(diag(confusionRatio), 2))

# Filter subtypes with >= 20 cells
df_num <- as.data.frame(table(non_ht_seurat_clean$non_ht_subtype))
select_cell <- as.character(df_num[df_num$Freq >= 20, ]$Var1)
select_cell <- intersect(select_cell, rownames(confusionRatio))
mat_to_plot <- confusionRatio[select_cell, select_cell, drop = FALSE]

# 8.1 With-values version
num_labels <- matrix("", nrow = nrow(mat_to_plot), ncol = ncol(mat_to_plot))
diag(num_labels) <- sprintf("%.2f", diag(mat_to_plot))

png(paste0(output_path, "nonHT_Ranger_Heatmap_withNum.png"), width = 1200, height = 1000, res = 150)
pheatmap(mat_to_plot, breaks = seq(0, 1, 0.01), cluster_rows = FALSE, cluster_cols = FALSE,
         border_color = NA, cellwidth = 45, cellheight = 45, fontsize = 14,
         color = viridis(100), main = "non-5-HT Subtypes Ranger Confusion Matrix",
         display_numbers = num_labels, fontsize_number = 12, number_color = "white")
dev.off()

# 8.2 No-values version
png(paste0(output_path, "nonHT_Ranger_Heatmap_noNum.png"), width = 1200, height = 1000, res = 150)
pheatmap(mat_to_plot, breaks = seq(0, 1, 0.01), cluster_rows = FALSE, cluster_cols = FALSE,
         border_color = NA, cellwidth = 45, cellheight = 45, fontsize = 14,
         color = viridis(100), main = "non-5-HT Subtypes Ranger Confusion Matrix",
         display_numbers = FALSE)
dev.off()

# On-screen display, no-values version
pheatmap(mat_to_plot, breaks = seq(0, 1, 0.01), cluster_rows = FALSE, cluster_cols = FALSE,
         border_color = NA, cellwidth = 45, cellheight = 45, fontsize = 14,
         color = viridis(100), main = "non-5-HT Subtypes Ranger Confusion Matrix (No Numbers)",
         display_numbers = FALSE)

cat("Ranger heatmaps saved\n")

cat("\n===== non-5-HT analysis completed =====\n")

In [ ]:
# ============================================
# 6.3 SLC17A6 / SLC17A8 / GAD1 / GAD2 violin plots per subtype
# ============================================
library(tidyr)   # pivot_longer lives here
vln_genes <- c("SLC17A6", "SLC17A8", "GAD1", "GAD2", "TH")
vln_genes <- vln_genes[vln_genes %in% rownames(non_ht_seurat_clean)]

expr_vln <- FetchData(non_ht_seurat_clean, vars = c("non_ht_subtype", vln_genes))

expr_vln_long <- expr_vln %>%
  pivot_longer(cols = -non_ht_subtype,
               names_to = "gene", values_to = "Expression") %>%
  mutate(gene = factor(gene, levels = vln_genes),
         non_ht_subtype = factor(non_ht_subtype))   # Add levels = ... here to fix the order

p_vln <- ggplot(expr_vln_long,
                aes(x = non_ht_subtype, y = Expression, fill = non_ht_subtype)) +
  geom_violin(scale = "width", trim = TRUE) +
  geom_boxplot(width = 0.15, outlier.size = 0.15, alpha = 0.7, color = "grey30") +
  facet_wrap(~ gene, nrow = 1, scales = "free_y") +
  labs(title = "SLC17A6 / SLC17A8 / GAD1 / GAD2: expression across non-5-HT subtypes",
       x = NULL, y = "Normalized Expression") +
  theme_bw(base_size = 13) +
  theme(legend.position = "none",
        panel.grid = element_blank(),
        strip.text = element_text(face = "bold", size = 12),
        axis.text.x = element_text(angle = 45, hjust = 1, size = 9))

print(p_vln)
ggsave(paste0(output_path, "nonHT_VlnPlot_Glut_GABA_markers.png"), p_vln,
       width = 14, height = 5, dpi = 300)
cat("Specified-gene violin plots saved\n")

In [ ]:


# # # ============================================library(Seurat)
library(dplyr)

save_path <- "./data/"

# ============================================
# 1. Define neurotransmitter type mapping (nonHT_15 reassigned to Glut)
# ============================================
glut_map <- c(
  "nonHT_0"  = "Glut_1",
  "nonHT_9" = "Glut_2",
  "nonHT_10" = "Glut_3", 
  "nonHT_17" = "Glut_4",
  "nonHT_22" = "Glut_5",   
  "nonHT_24" = "Glut_6",   
  "nonHT_25" = "Glut_7"
)

glut_gaba_map <- c(
  "nonHT_0"  = "Glut-GABA_1",
  "nonHT_4"  = "Glut-GABA_2",
  "nonHT_8"  = "Glut-GABA_3",
  "nonHT_11"  = "Glut-GABA_4",
  "nonHT_13"  = "Glut-GABA_5",
  "nonHT_28"  = "Glut-GABA_6"
)

gaba_map <- c(
  "nonHT_1"  = "GABA_1",
  "nonHT_2"  = "GABA_2",
  "nonHT_3"  = "GABA_3",
  "nonHT_5"  = "GABA_4",
  "nonHT_6"  = "GABA_5",
  "nonHT_7"  = "GABA_6",
  "nonHT_12" = "GABA_7",
  "nonHT_14" = "GABA_8",
  "nonHT_15" = "GABA_9",
  "nonHT_16" = "GABA_10",
  "nonHT_18" = "GABA_11",
  "nonHT_19" = "GABA_12",
  "nonHT_20" = "GABA_13",
  "nonHT_21" = "GABA_14",
  "nonHT_23" = "GABA_15",
  "nonHT_27" = "GABA_16"
)

da_map <- c("nonHT_26" = "DA_1")

# Merge all mappings
all_map <- c(glut_map, glut_gaba_map, gaba_map, da_map)

cat("Mapping table:\n")
print(all_map)

# ============================================
# Fix: rename neuro_type
# ============================================

# Look up table in cell order
neuro_vec <- all_map[as.character(non_ht_seurat_clean$non_ht_subtype)]

# Key: drop names, otherwise Seurat matches cells by name
names(neuro_vec) <- NULL

# Assign
non_ht_seurat_clean$neuro_type <- neuro_vec

# Fix factor order
type_order <- c(
  paste0("Glut_", 1:7),
  paste0("Glut-GABA_", 1:6),
  paste0("GABA_", 1:16),
  "DA_1"
)
non_ht_seurat_clean$neuro_type <- factor(non_ht_seurat_clean$neuro_type, levels = type_order)

cat("\nnon-5-HT neurotransmitter type distribution:\n")
print(table(non_ht_seurat_clean$neuro_type))

# # # 4. Save
# # # ============================================
# # saveRDS(non_ht_seurat_clean, paste0(save_path, "non_ht_seurat_neurotype.rds"))
# # saveRDS(classFiltSeurat, paste0(save_path, "classFiltSeurat_neurotype.rds"))

# # cat("\nOK: fix completed and saved!\n")

In [ ]:
# ============================================
# 3. Synchronize update to classFiltSeurat
# ============================================

# First convert factor to character (key! otherwise new types become NA)
classFiltSeurat$cell_type <- as.character(classFiltSeurat$cell_type)

# Extract non-5-HT cell names
nonht_cells <- colnames(non_ht_seurat_clean)

# Assign (no more NA now)
classFiltSeurat$cell_type[nonht_cells] <- as.character(non_ht_seurat_clean$neuro_type)

# Reorganize factor levels for all cells
all_types <- unique(classFiltSeurat$cell_type)

ht_types       <- sort(grep("^5-HT", all_types, value = TRUE))
glut_types     <- sort(grep("^Glut_", all_types, value = TRUE))
glutgaba_types <- sort(grep("^Glut-GABA_", all_types, value = TRUE))
gaba_types     <- sort(grep("^GABA_", all_types, value = TRUE))
da_types       <- sort(grep("^DA_", all_types, value = TRUE))
other_types    <- sort(setdiff(all_types, c(ht_types, glut_types, glutgaba_types, gaba_types, da_types)))

type_levels <- c(ht_types, glut_types, glutgaba_types, gaba_types, da_types, other_types)
classFiltSeurat$cell_type <- factor(classFiltSeurat$cell_type, levels = type_levels)

cat("\nclassFiltSeurat cell_type distribution of all cells:\n")
print(table(classFiltSeurat$cell_type))



In [ ]:
# ============================================
# Remove cells whose cell_type starts with "Neurons_"
# ============================================

# First check how many will be deleted
neuron_idx <- grepl("^Neurons_", classFiltSeurat$cell_type)
cat("Neurons_* cells to be removed:", sum(neuron_idx), "\n")
cat("Total cells before removal:", ncol(classFiltSeurat), "\n")

# Subset out these cells
classFiltSeurat <- subset(classFiltSeurat, subset = cell_type %in% 
                            levels(classFiltSeurat$cell_type)[!grepl("^Neurons_", levels(classFiltSeurat$cell_type))])

# Or the more direct equivalent:
# classFiltSeurat <- subset(classFiltSeurat, cells = colnames(classFiltSeurat)[!neuron_idx])

cat("Total cells after removal:", ncol(classFiltSeurat), "\n")

# Reorganize factor levels (drop empty levels)
classFiltSeurat$cell_type <- droplevels(classFiltSeurat$cell_type)

cat("\ncell_type distribution after removal:\n")
print(table(classFiltSeurat$cell_type))

In [ ]:
# ============================================
# 4. Save
# ============================================
saveRDS(non_ht_seurat_clean, paste0(save_path, "non_ht_seurat_neurotype.rds"))
saveRDS(classFiltSeurat, paste0(save_path, "classFiltSeurat_neurotype.rds"))

cat("\nOK: fix completed and saved!\n")

In [ ]:
library(Seurat)
library(ggplot2)
library(dplyr)

save_path <- "./data/"

# ============================================
# Read in both objects
# ============================================
non_ht_seurat_clean <- readRDS(paste0(save_path, "non_ht_seurat_neurotype.rds"))
# classFiltSeurat     <- readRDS(paste0(save_path, "classFiltSeurat_neurotype.rds"))

# ============================================
# Check after loading (confirm correct objects)
# ============================================
cat("===== non_ht_seurat_clean =====\n")
print(non_ht_seurat_clean)                      # Assay / reduction overview
cat("Cells:", ncol(non_ht_seurat_clean), "\n")
cat("neuro_type distribution:\n")
print(table(non_ht_seurat_clean$neuro_type))

# cat("\n===== classFiltSeurat =====\n")
# print(classFiltSeurat)
# cat("Cells:", ncol(classFiltSeurat), "\n")
# cat("cell_type_fine distribution:\n")
# print(table(classFiltSeurat$cell_type_fine))

In [ ]:
library(Seurat)
library(ggplot2)
library(dplyr)
library(ranger)
library(pheatmap)
library(viridis)

output_path <- "./data/"

# ============================================
# 0. Confirm data & factor order
# ============================================
# Make sure neuro_type exists and is a factor
if(!"neuro_type" %in% colnames(non_ht_seurat_clean@meta.data)) {
  stop("Please run the neurotransmitter type renaming code first!")
}

type_order <- c(
  paste0("Glut_", 1:7),
  paste0("Glut-GABA_", 1:6),
  paste0("GABA_", 1:16),
  "DA_1"
)
non_ht_seurat_clean$neuro_type <- factor(non_ht_seurat_clean$neuro_type, levels = type_order)
non_ht_seurat_clean <- SetIdent(non_ht_seurat_clean, value = "neuro_type")

cat("non-5-HT neurotransmitter type distribution:\n")
print(table(non_ht_seurat_clean$neuro_type))

# ============================================
# 1. UMAP plot (by neurotransmitter type)
# ============================================
# 1.1 Labeled version
p_neuro_umap <- DimPlot(non_ht_seurat_clean, group.by = "neuro_type",
                        label = TRUE, repel = TRUE, raster = FALSE) +
  coord_fixed() +
  ggtitle("non-5-HT Neurons by Neurotransmitter Type") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"))
print(p_neuro_umap)

ggsave(paste0(output_path, "nonHT_NeuroType_UMAP_labeled.png"),
       p_neuro_umap, width = 12, height = 10, dpi = 300)
ggsave(paste0(output_path, "nonHT_NeuroType_UMAP_labeled.pdf"),   # Newly added
       p_neuro_umap, width = 12, height = 10, limitsize = FALSE)

# 1.2 No-label version
p_neuro_umap_nolabel <- DimPlot(non_ht_seurat_clean, group.by = "neuro_type",
                                label = FALSE, raster = FALSE) +
  coord_fixed() +
  ggtitle("non-5-HT Neurons by Neurotransmitter Type (No Label)") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"),
        legend.position = "right")
print(p_neuro_umap_nolabel)

ggsave(paste0(output_path, "nonHT_NeuroType_UMAP_nolabel.png"),
       p_neuro_umap_nolabel, width = 12, height = 10, dpi = 300)
ggsave(paste0(output_path, "nonHT_NeuroType_UMAP_nolabel.pdf"),   # Newly added
       p_neuro_umap_nolabel, width = 12, height = 10, limitsize = FALSE)

cat("UMAP plots saved (png + pdf)\n")


In [ ]:
# ============================================
# 2. Find marker genes (Wilcoxon) - grouped by neuro_type, run only once
# ============================================
nonht_markers <- FindAllMarkers(non_ht_seurat_clean,
                                only.pos = TRUE,
                                min.pct = 0.25,
                                logfc.threshold = 0.25,
                                test.use = "wilcox")

# Save Top20
top20 <- nonht_markers %>%
  group_by(cluster) %>%
  top_n(n = 20, wt = avg_log2FC) %>%
  arrange(cluster, desc(avg_log2FC))

write.csv(top20, paste0(output_path, "nonHT_NeuroType_top20_markers.csv"), row.names = FALSE)
cat("Top20 markers saved\n")

# ============================================
# 3. Top5 DotPlot (by neurotransmitter type)
# ============================================
top5_genes <- nonht_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 5) %>%
  ungroup() %>%
  arrange(cluster)

gene_order5 <- top5_genes %>% arrange(cluster) %>% pull(gene) %>% unique()

p_dot5 <- DotPlot(non_ht_seurat_clean, features = gene_order5, group.by = "neuro_type",
                  dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("non-5-HT neurotransmitter types: Top5 marker gene dot plot") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 8),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_dot5)
ggsave(paste0(output_path, "nonHT_NeuroType_Top5_DotPlot.png"), p_dot5, 
       width = 16, height = 18, dpi = 300)

write.csv(top5_genes %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "nonHT_NeuroType_Top5_table.csv"), row.names = FALSE)
cat("Top5 dot plot saved\n")

# ============================================
# 4. Top10 DotPlot
# ============================================
top10_genes <- nonht_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 10) %>%
  ungroup() %>%
  arrange(cluster)

gene_order10 <- top10_genes %>% arrange(cluster) %>% pull(gene) %>% unique()

p_dot10 <- DotPlot(non_ht_seurat_clean, features = gene_order10, group.by = "neuro_type",
                   dot.scale = 4, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("non-5-HT neurotransmitter types: Top10 marker gene dot plot") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 7),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_dot10)
ggsave(paste0(output_path, "nonHT_NeuroType_Top10_DotPlot.png"), p_dot10, 
       width = 18, height = 24, dpi = 300)

write.csv(top10_genes %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "nonHT_NeuroType_Top10_table.csv"), row.names = FALSE)
cat("Top10 dot plot saved\n")

# ============================================
# 5. Top20 DotPlot
# ============================================
gene_order20 <- top20 %>% arrange(cluster) %>% pull(gene) %>% unique()

p_dot20 <- DotPlot(non_ht_seurat_clean, features = gene_order20, group.by = "neuro_type",
                   dot.scale = 3, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("non-5-HT neurotransmitter types: Top20 marker gene dot plot") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 5.5),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )
print(p_dot20)
ggsave(paste0(output_path, "nonHT_NeuroType_Top20_DotPlot.png"), p_dot20, 
       width = 20, height = 34, dpi = 300)

write.csv(top20 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "nonHT_NeuroType_Top20_table.csv"), row.names = FALSE)
cat("Top20 dot plot saved\n")

In [ ]:
# OTOF,NPTX1;SLC17A8,LMX1B;ADGRD2,CDH5;TRIM55,IRX4;PRDM6,LHX9;FST,BCHE;SEMA3A,SCTR;PENK.TACR3;SKOR2,CCDC60;CCK,RIN2;ATP12A,PGM5;SLC32A1,GSTP1;SLC6A5,LAMP5;FGFR2,SATB1;OTX2,SERPINF1;GALR1,SYT6;TAC3,IL13RA1;GLI3,GATA3;TLL1,ADAMTSL1;GSN,CXCL14;FLT1,ONECUT1;SNTG2,CHRNA2;KIF25,BCL11B;SST,CHST9;FOXA2,EMX2;CPA4,RLN3;TH,SLC6A2


In [ ]:
library(Seurat)
library(ggplot2)

output_path <- "./data/"

# ============================================
# 0. Confirm merged neuro_type order
# ============================================
type_order <- c(
  paste0("Glut_", 1:7),
  paste0("Glut-GABA_", 1:6),
  paste0("GABA_", 1:16),
  "DA_1"
)
non_ht_seurat_clean$neuro_type <- factor(non_ht_seurat_clean$neuro_type, levels = type_order)

# ============================================
# 1. New gene list (28 pairs, expanded in order)
# ============================================
gene_list <- c("OTOF","NPTX1","SLC17A8","LMX1B","ADGRD2","CDH5","TRIM55","IRX4",
               "PRDM6","LHX9","FST","BCHE","SEMA3A","SCTR","PENK","TACR3",
               "SKOR2","CCDC60","CCK","RIN2","ATP12A","PGM5","SLC32A1","GSTP1",
               "SLC6A5","LAMP5","FGFR2","SATB1","OTX2","SERPINF1","GALR1","SYT6",
               "TAC3","IL13RA1","GLI3","GATA3","TLL1","ADAMTSL1","GSN","CXCL14",
               "FLT1","ONECUT1","SNTG2","CHRNA2","KIF25","BCL11B","SST","CHST9",
               "FOXA2","EMX2","CPA4","RLN3","TH","SLC6A2")

available_genes <- gene_list[gene_list %in% rownames(non_ht_seurat_clean)]
missing_genes   <- gene_list[!gene_list %in% rownames(non_ht_seurat_clean)]

cat("Total genes:", length(gene_list), ", available:", length(available_genes), "\n")
if(length(missing_genes) > 0) {
  cat("WARNING: not found, skipped:", paste(missing_genes, collapse = ", "), "\n")
}

# ============================================
# 2. DotPlot
# ============================================
p_custom <- DotPlot(
  non_ht_seurat_clean,
  features = available_genes,
  group.by = "neuro_type",
  dot.scale = 6,
  cols = c("lightgrey", "red")
) +
  coord_flip() +
  RotatedAxis() +
  ggtitle("non-5-HT neurotransmitter types: selected markers") +
  xlab("Neurotransmitter type") +
  ylab("Gene") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 10, face = "bold"),
    axis.text.y = element_text(size = 9),
    axis.title  = element_text(size = 12, face = "bold"),
    plot.title  = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid  = element_blank()
  )

print(p_custom)

ggsave(
  filename = paste0(output_path, "nonHT_NeuroType_Merged_markers_DotPlot.png"),
  plot = p_custom,
  width = 16,
  height = 16,
  dpi = 300
)

# Also save a PDF version (vector graphic, resolution-independent)
ggsave(
  filename = paste0(output_path, "nonHT_NeuroType_Merged_markers_DotPlot.pdf"),
  plot = p_custom,
  width = 16,
  height = 16
)

cat("PNG and PDF both saved!\n")

In [ ]:
# ============================================
# 6. Specified-gene DotPlot + UMAP FeaturePlot
# ============================================
gene_list_input <- c('SLC17A6','SLC17A7','SLC17A8','GAD1','GAD2','TH')
gene_list <- toupper(gene_list_input)
gene_list[gene_list == "CHE"] <- "ACHE"

available_genes <- gene_list[gene_list %in% rownames(non_ht_seurat_clean)]
missing_genes <- gene_list[!gene_list %in% rownames(non_ht_seurat_clean)]

if(length(missing_genes) > 0) {
  cat("WARNING: the following genes were not found and skipped:", paste(missing_genes, collapse = ", "), "\n\n")
}
cat("OK: available genes (", length(available_genes), "):\n")
cat(paste(available_genes, collapse = ", "), "\n\n")

# 6.1 DotPlot (by neuro_type)
p_custom <- DotPlot(non_ht_seurat_clean, features = available_genes, group.by = "neuro_type",
                    dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("non-5-HT neurotransmitter types: specified marker gene dot plot") +
  xlab("Neurotransmitter type") + ylab("Gene") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 10, face = "bold"),
    axis.text.y = element_text(size = 10),
    axis.title = element_text(size = 12, face = "bold"),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )
print(p_custom)
ggsave(paste0(output_path, "nonHT_NeuroType_Custom_DotPlot.png"), p_custom, 
       width = 14, height = 10, dpi = 300)
cat("Specified-gene dot plot saved\n")

# # 6.2 UMAP FeaturePlot
# p_features <- FeaturePlot(non_ht_seurat_clean, features = available_genes, 
#                           ncol = 3, raster = FALSE, cols = c("lightgrey", "red"))
# print(p_features)
# ggsave(paste0(output_path, "nonHT_NeuroType_Custom_UMAP.png"), p_features, 
#        width = 15, height = 12, dpi = 300)
# cat("Specified-gene UMAP FeaturePlot saved\n")

In [ ]:
library(viridis)
library(ranger)
library(pheatmap)
library(grid)

output_path <- "./data/"

# ============================================
# 0. Confirm merged neuro_type order
# ============================================
type_order <- c(
  paste0("Glut_", 1:7),
  paste0("Glut-GABA_", 1:6),
  paste0("GABA_", 1:16),
  "DA_1"
)
non_ht_seurat_clean$neuro_type <- factor(non_ht_seurat_clean$neuro_type, levels = type_order)

# ============================================
# 1. New gene list (28 pairs, expanded in order)
# ============================================
gene_list <- c("OTOF","NPTX1","SLC17A8","LMX1B","ADGRD2","CDH5","TRIM55","IRX4",
               "PRDM6","LHX9","FST","BCHE","SEMA3A","SCTR","PENK","TACR3",
               "SKOR2","CCDC60","CCK","RIN2","ATP12A","PGM5","SLC32A1","GSTP1",
               "SLC6A5","LAMP5","FGFR2","SATB1","OTX2","SERPINF1","GALR1","SYT6",
               "TAC3","IL13RA1","GLI3","GATA3","TLL1","ADAMTSL1","GSN","CXCL14",
               "FLT1","ONECUT1","SNTG2","CHRNA2","KIF25","BCL11B","SST","CHST9",
               "FOXA2","EMX2","CPA4","RLN3","TH","SLC6A2")

available_genes <- gene_list[gene_list %in% rownames(non_ht_seurat_clean)]
missing_genes   <- gene_list[!gene_list %in% rownames(non_ht_seurat_clean)]

cat("Total genes:", length(gene_list), ", available:", length(available_genes), "\n")
if(length(missing_genes) > 0) {
  cat("WARNING: not found, skipped:", paste(missing_genes, collapse = ", "), "\n")
}

# ============================================
# 2. DotPlot
# ============================================
p_custom <- DotPlot(
  non_ht_seurat_clean,
  features = available_genes,
  group.by = "neuro_type",
  dot.scale = 6,
  cols = c("lightgrey", "red")
) +
  coord_flip() +
  RotatedAxis() +
  ggtitle("non-5-HT neurotransmitter types: selected markers") +
  xlab("Neurotransmitter type") +
  ylab("Gene") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 10, face = "bold"),
    axis.text.y = element_text(size = 9),
    axis.title  = element_text(size = 12, face = "bold"),
    plot.title  = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid  = element_blank()
  )

print(p_custom)

ggsave(
  filename = paste0(output_path, "nonHT_NeuroType_Merged_markers_DotPlot.png"),
  plot = p_custom,
  width = 16,
  height = 16,
  dpi = 300
)
ggsave(
  filename = paste0(output_path, "nonHT_NeuroType_Merged_markers_DotPlot.pdf"),
  plot = p_custom,
  width = 16,
  height = 16
)
cat("DotPlot PNG + PDF saved!\n")

# ============================================
# 7. Violin plots (by neurotransmitter type)
# ============================================
plot_data <- non_ht_seurat_clean@meta.data
plot_data$neuro_type <- factor(plot_data$neuro_type, levels = type_order)

# 7.1 nFeature_RNA (ticks every 1000)
nfeature_max <- max(plot_data$nFeature_RNA, na.rm = TRUE)
nfeature_breaks <- seq(0, ceiling(nfeature_max / 1000) * 1000, by = 1000)

p_nfeature <- ggplot(plot_data, aes(x = neuro_type, y = nFeature_RNA, fill = neuro_type)) +
  geom_violin(scale = "width", trim = TRUE, width = 0.7, linewidth = 0.3, color = "black") +
  scale_y_continuous(breaks = nfeature_breaks) +
  labs(x = "", y = "nFeature RNA") +
  theme_bw() +
  theme(
    panel.grid.major = element_blank(), panel.grid.minor = element_blank(),
    panel.border = element_blank(), axis.line = element_line(color = "black"),
    axis.ticks = element_line(color = "black"), axis.ticks.length = unit(6, "pt"),
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 10), axis.title = element_text(size = 13),
    legend.position = "none"
  )
print(p_nfeature)

ggsave(paste0(output_path, "violin_nonHT_NeuroType_nFeature_RNA_1k.png"),
       p_nfeature, width = 10, height = 6, dpi = 150)
ggsave(paste0(output_path, "violin_nonHT_NeuroType_nFeature_RNA_1k.pdf"),
       p_nfeature, width = 14, height = 8)

# 7.2 nCount_RNA
p_ncount <- ggplot(plot_data, aes(x = neuro_type, y = nCount_RNA, fill = neuro_type)) +
  geom_violin(scale = "width", trim = TRUE, width = 0.7, linewidth = 0.3, color = "black") +
  labs(x = "", y = "nCount RNA") +
  theme_bw() +
  theme(
    panel.grid.major = element_blank(), panel.grid.minor = element_blank(),
    panel.border = element_blank(), axis.line = element_line(color = "black"),
    axis.ticks = element_line(color = "black"), axis.ticks.length = unit(6, "pt"),
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 10), axis.title = element_text(size = 13),
    legend.position = "none"
  )
print(p_ncount)

ggsave(paste0(output_path, "violin_nonHT_NeuroType_nCount_RNA.png"),
       p_ncount, width = 10, height = 6, dpi = 150)
ggsave(paste0(output_path, "violin_nonHT_NeuroType_nCount_RNA.pdf"),
       p_ncount, width = 14, height = 8)
cat("Violin plots PNG + PDF saved\n")

# ============================================
# 8. Ranger + heatmap (by neuro_type)
# ============================================
expr_matrix <- GetAssayData(non_ht_seurat_clean, layer = "data")
genes <- VariableFeatures(non_ht_seurat_clean)

X <- t(as.matrix(expr_matrix[genes, , drop = FALSE]))
y <- factor(non_ht_seurat_clean$neuro_type)

rf_nonht <- ranger(x = X, y = y, num.trees = 500, num.threads = 8, classification = TRUE)

cat("\nOOB Error:", rf_nonht$prediction.error, "\n")

conf_mat <- rf_nonht$confusion.matrix
confusionRatio <- t(apply(conf_mat, 1, function(x) {
  if(sum(x) == 0) return(rep(0, length(x)))
  x / sum(x)
}))

cat("Diagonal accuracy:\n")
print(round(diag(confusionRatio), 2))

# Filter subtypes with >= 20 cells
df_num <- as.data.frame(table(non_ht_seurat_clean$neuro_type))
select_cell <- as.character(df_num[df_num$Freq >= 20, ]$Var1)
select_cell <- intersect(select_cell, rownames(confusionRatio))
mat_to_plot <- confusionRatio[select_cell, select_cell, drop = FALSE]

# 8.1 With-values version
num_labels <- matrix("", nrow = nrow(mat_to_plot), ncol = ncol(mat_to_plot))
diag(num_labels) <- sprintf("%.2f", diag(mat_to_plot))

# --- PNG ---
png(paste0(output_path, "nonHT_NeuroType_Ranger_Heatmap_withNum.png"), width = 1400, height = 1200, res = 150)
pheatmap(mat_to_plot, breaks = seq(0, 1, 0.01), cluster_rows = FALSE, cluster_cols = FALSE,
         border_color = NA, cellwidth = 40, cellheight = 40, fontsize = 12,
         color = viridis(100), main = "non-5-HT Neurotransmitter Types Ranger Confusion Matrix",
         display_numbers = num_labels, fontsize_number = 10, number_color = "white")
dev.off()

# --- PDF ---
pdf(paste0(output_path, "nonHT_NeuroType_Ranger_Heatmap_withNum.pdf"),
    width = 14, height = 12)  # Units are inches
pheatmap(mat_to_plot, breaks = seq(0, 1, 0.01), cluster_rows = FALSE, cluster_cols = FALSE,
         border_color = NA, cellwidth = 40, cellheight = 40, fontsize = 12,
         color = viridis(100), main = "non-5-HT Neurotransmitter Types Ranger Confusion Matrix",
         display_numbers = num_labels, fontsize_number = 10, number_color = "white")
dev.off()

# 8.2 No-values version
# --- PNG ---
png(paste0(output_path, "nonHT_NeuroType_Ranger_Heatmap_noNum.png"), width = 1400, height = 1200, res = 150)
pheatmap(mat_to_plot, breaks = seq(0, 1, 0.01), cluster_rows = FALSE, cluster_cols = FALSE,
         border_color = NA, cellwidth = 40, cellheight = 40, fontsize = 12,
         color = viridis(100), main = "non-5-HT Neurotransmitter Types Ranger Confusion Matrix",
         display_numbers = FALSE)
dev.off()

# --- PDF ---
pdf(paste0(output_path, "nonHT_NeuroType_Ranger_Heatmap_noNum.pdf"),
    width = 14, height = 12)
pheatmap(mat_to_plot, breaks = seq(0, 1, 0.01), cluster_rows = FALSE, cluster_cols = FALSE,
         border_color = NA, cellwidth = 40, cellheight = 40, fontsize = 12,
         color = viridis(100), main = "non-5-HT Neurotransmitter Types Ranger Confusion Matrix",
         display_numbers = FALSE)
dev.off()

# On-screen display, no-values version
pheatmap(mat_to_plot, breaks = seq(0, 1, 0.01), cluster_rows = FALSE, cluster_cols = FALSE,
         border_color = NA, cellwidth = 40, cellheight = 40, fontsize = 12,
         color = viridis(100), main = "non-5-HT Neurotransmitter Types Ranger (No Numbers)",
         display_numbers = FALSE)

cat("Ranger heatmaps saved\n")

cat("\n===== non-5-HT neurotransmitter type analysis completed =====\n")

In [ ]:
library(viridis)
library(grid)

# ============================================
# 8. Ranger + heatmap (by neuro_type) - complete version
# ============================================
expr_matrix <- GetAssayData(non_ht_seurat_clean, layer = "data")
genes <- VariableFeatures(non_ht_seurat_clean)

X <- t(as.matrix(expr_matrix[genes, , drop = FALSE]))
y <- factor(non_ht_seurat_clean$neuro_type)

rf_nonht <- ranger(x = X, y = y, num.trees = 500, num.threads = 8, classification = TRUE)

cat("\nOOB Error:", rf_nonht$prediction.error, "\n")

conf_mat <- rf_nonht$confusion.matrix
confusionRatio <- t(apply(conf_mat, 1, function(x) {
  if(sum(x) == 0) return(rep(0, length(x)))
  x / sum(x)
}))

cat("Diagonal accuracy:\n")
print(round(diag(confusionRatio), 2))

cat("\nCell counts for all neuro_type:\n")
print(table(non_ht_seurat_clean$neuro_type))

# ---- Keep all types (threshold 10) ----
df_num <- as.data.frame(table(non_ht_seurat_clean$neuro_type))
select_cell <- as.character(df_num[df_num$Freq >= 10, ]$Var1)
select_cell <- intersect(select_cell, rownames(confusionRatio))

type_order_present <- type_order[type_order %in% select_cell]
mat_to_plot <- confusionRatio[type_order_present, type_order_present, drop = FALSE]

cat("\nNumber of types in heatmap:", nrow(mat_to_plot), "\n")
cat("Excluded types:", setdiff(type_order, type_order_present), "\n")

# ---- Numeric labels ----
num_labels <- matrix("", nrow = nrow(mat_to_plot), ncol = ncol(mat_to_plot))
diag(num_labels) <- sprintf("%.2f", diag(mat_to_plot))

# ============================================
# Core: save at actual gtable size, never truncate
# ============================================
save_pheatmap_exact <- function(ph_obj, file_base, dpi = 150) {

  w_in <- convertWidth(sum(ph_obj$gtable$widths),  "inch", valueOnly = TRUE)
  h_in <- convertHeight(sum(ph_obj$gtable$heights), "inch", valueOnly = TRUE)

  cat(sprintf("Actual size: %.1f x %.1f inch\n", w_in, h_in))

  # --- PNG ---
  png(paste0(file_base, ".png"),
      width = w_in, height = h_in, units = "in", res = dpi)
  grid.newpage()
  grid.draw(ph_obj$gtable)
  dev.off()

  # --- PDF ---
  pdf(paste0(file_base, ".pdf"), width = w_in, height = h_in)
  grid.newpage()
  grid.draw(ph_obj$gtable)
  dev.off()

  cat("OK: saved:", file_base, "(PNG + PDF)\n")
}

# ---- 8.1 With-values version ----
p_num <- pheatmap(mat_to_plot,
                  breaks = seq(0, 1, 0.01),
                  cluster_rows = FALSE,
                  cluster_cols = FALSE,
                  border_color = NA,
                  cellwidth = 30,
                  cellheight = 30,
                  fontsize = 10,
                  color = viridis(100),
                  main = "non-5-HT Neurotransmitter Types Ranger Confusion Matrix",
                  display_numbers = num_labels,
                  fontsize_number = 8,
                  number_color = "white")

save_pheatmap_exact(p_num, paste0(output_path, "nonHT_NeuroType_Ranger_Heatmap_withNum"))

# ---- 8.2 No-values version ----
p_nonum <- pheatmap(mat_to_plot,
                    breaks = seq(0, 1, 0.01),
                    cluster_rows = FALSE,
                    cluster_cols = FALSE,
                    border_color = NA,
                    cellwidth = 30,
                    cellheight = 30,
                    fontsize = 10,
                    color = viridis(100),
                    main = "non-5-HT Neurotransmitter Types Ranger Confusion Matrix",
                    display_numbers = FALSE)

save_pheatmap_exact(p_nonum, paste0(output_path, "nonHT_NeuroType_Ranger_Heatmap_noNum"))

# ---- On-screen display ----
grid.newpage()
grid.draw(p_nonum$gtable)

cat("Ranger heatmaps saved\n")

In [ ]:
library(Seurat)

cat("===== Loading saved Seurat object =====\n")

save_path <- "./data/"

classFiltSeurat_filt <- readRDS(
  paste0(save_path, "all_seurat_with_newHT_res05.rds")
)

cat("Loaded successfully, total cells:", ncol(classFiltSeurat_filt), "\n")
cat("Total genes:", nrow(classFiltSeurat_filt), "\n")

cat("\ncell_type distribution:\n")
print(table(classFiltSeurat_filt$cell_type))

if ("major_group" %in% colnames(classFiltSeurat_filt@meta.data)) {
  cat("\nmajor_group distribution:\n")
  print(table(classFiltSeurat_filt$major_group))
}

cat("\nObject loaded into variable: classFiltSeurat_filt\n")

In [ ]:
# Cluster non-neuronal populations separately

In [ ]:
library(Seurat)
library(ggplot2)

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(clustree)
library(harmony)

output_path <- "./data/"

save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("OK: saved:", file_base, "(PNG + PDF)\n")
}

# ============================================
# 0. Read full object + extract Oligo
# ============================================
# all_seurat <- readRDS(paste0(output_path, "classFiltSeurat_neurotype.rds"))

cat("===== Re-clustering Astrocytes =====\n")

# ---------- 1. Extract Astrocytes ----------
astro_cells <- colnames(all_seurat)[all_seurat$cell_type == "Astrocytes"]
astro_seurat <- subset(all_seurat, cells = astro_cells)
cat("Astrocytes cells:", ncol(astro_seurat), "\n")
print(table(astro_seurat$donor))

# ---------- 2. RNA normalization + PCA ----------
DefaultAssay(astro_seurat) <- "RNA"
astro_seurat <- NormalizeData(astro_seurat, verbose = FALSE)
astro_seurat <- FindVariableFeatures(astro_seurat, selection.method = "vst",
                                     nfeatures = 2000, verbose = FALSE)
astro_seurat <- ScaleData(astro_seurat, verbose = FALSE)
astro_seurat <- RunPCA(astro_seurat, features = VariableFeatures(astro_seurat),
                       npcs = 50, verbose = FALSE)

# ---------- 3. Harmony (donor batch correction, consistent with 5-HT pipeline) ----------
astro_seurat <- RunHarmony(astro_seurat,
                           group.by.vars = "donor",
                           reduction.use = "pca",
                           dims.use = 1:30,
                           verbose = FALSE)

# ---------- 4. Neighbor graph + multi-resolution clustering + clustree ----------
astro_seurat <- FindNeighbors(astro_seurat, reduction = "harmony", dims = 1:30,
                              verbose = FALSE)

resolutions_to_try <- c(0.1, 0.2, 0.3, 0.4, 0.5, 0.8, 1.0)
for (res in resolutions_to_try) {
  astro_seurat <- FindClusters(astro_seurat, resolution = res,
                               future.seed = TRUE, verbose = FALSE)
}

p_clustree <- clustree(astro_seurat, prefix = "RNA_snn_res.")
print(p_clustree)
save_fig(p_clustree, paste0(output_path, "astro_clustree"), 12, 9)

cat("\nClustering summary at each resolution:\n")
for (res in resolutions_to_try) {
  tab <- table(astro_seurat@meta.data[[paste0("RNA_snn_res.", res)]])
  cat(sprintf("res=%.1f: %d clusters, smallest cluster %d cells\n",
              res, length(tab), min(tab)))
}

